# XGBoost /CatgBoost Model — Exploratory Data Analysis
## Predicting Player Performance Decline Across Multiple Competitions
### Seasons: 2022-2023 · 2023-2024 · 2024-2025

**Goal**: Identify and combine all available data (fixtures, rest days, congestion, minutes played, per-match stats, injury impact) to build a dataset for predicting performance decline in players competing in multiple competitions — with focus on Champions League teams.


## Section 1 — Setup & Directory Inventory
Scan the full `/data` directory and catalogue every CSV file available.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = "."

SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

RAW_SEASON_DIRS = {
    "2022-2023": os.path.join(BASE_DIR, "2022-2023"),
    "2023-2024": os.path.join(BASE_DIR, "2023-2024"),
    "2024-2025": os.path.join(BASE_DIR, "2024-2025"),
}

API_SEASON_DIRS = {
    "2022-2023": os.path.join(BASE_DIR, "API_SEASON_2022_2023"),
    "2023-2024": os.path.join(BASE_DIR, "API_SEASON_2023_2024"),
    "2024-2025": os.path.join(BASE_DIR, "API_SEASON_2024_2025"),
}

PROCESSED_SEASON_DIRS = {
    "2022-2023": os.path.join(BASE_DIR, "SEASON_2022_2023"),
    "2023-2024": os.path.join(BASE_DIR, "SEASON_2023_2024"),
    "2024-2025": os.path.join(BASE_DIR, "SEASON_2024_2025"),
}

FINAL_CSV = os.path.join(BASE_DIR, "Fixture_IQ_Data_Seasons_2022-2025.csv")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def scan_csv_file(path, season=None, source_group=None):
    """
    Reads only the header and basic shape of a CSV.
    This is safe and does not modify anything.
    """
    info = {
        "season": season,
        "source_group": source_group,
        "file_name": os.path.basename(path),
        "file_path": path,
        "folder": os.path.dirname(path),
        "size_kb": round(os.path.getsize(path) / 1024, 1),
        "n_rows": np.nan,
        "n_cols": np.nan,
        "columns": None,
        "read_error": None,
    }

    try:
        df_tmp = pd.read_csv(path, nrows=5)
        info["n_cols"] = len(df_tmp.columns)
        info["columns"] = list(df_tmp.columns)

        # Count rows safely without loading the full file into memory
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            info["n_rows"] = max(sum(1 for _ in f) - 1, 0)

    except Exception as e:
        info["read_error"] = str(e)

    return info


def find_csvs(folder):
    """
    Finds all CSV files recursively inside a folder.
    """
    if not os.path.exists(folder):
        return []
    return sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))


def classify_file(file_name, file_path):
    """
    Simple classification to identify likely data type.
    """
    text = (file_name + " " + file_path).lower()

    if "injur" in text or "absence" in text or "burden" in text:
        return "injuries"
    elif "player_stats" in text or "player" in text:
        return "player_performance"
    elif "team_results" in text or "team" in text:
        return "team_data"
    elif "fixture" in text or "match" in text:
        return "fixtures"
    elif "sofascore" in text:
        return "sofascore"
    elif "fbref" in text:
        return "fbref"
    else:
        return "other"


# ============================================================
# 1. CURRENT DIRECTORY INVENTORY
# ============================================================

print("=" * 100)
print("CURRENT WORKING DIRECTORY")
print("=" * 100)
print(os.getcwd())

print("\nFiles and folders in current directory:")
for item in sorted(os.listdir(BASE_DIR)):
    path = os.path.join(BASE_DIR, item)
    if os.path.isdir(path):
        print(f"📁 {item}/")
    else:
        print(f"📄 {item}")


# ============================================================
# 2. SCAN ALL CSV FILES
# ============================================================

all_file_infos = []

print("\n" + "=" * 100)
print("SCANNING RAW SEASON DIRECTORIES")
print("=" * 100)

for season, folder in RAW_SEASON_DIRS.items():
    print(f"\n📅 {season} | {folder}")

    csvs = find_csvs(folder)
    print(f"CSV files found: {len(csvs)}")

    for path in csvs:
        info = scan_csv_file(path, season=season, source_group="raw_season")
        info["data_type_guess"] = classify_file(info["file_name"], info["file_path"])
        all_file_infos.append(info)


print("\n" + "=" * 100)
print("SCANNING API SEASON DIRECTORIES")
print("=" * 100)

for season, folder in API_SEASON_DIRS.items():
    print(f"\n📅 {season} | {folder}")

    csvs = find_csvs(folder)
    print(f"CSV files found: {len(csvs)}")

    for path in csvs:
        info = scan_csv_file(path, season=season, source_group="api_season")
        info["data_type_guess"] = classify_file(info["file_name"], info["file_path"])
        all_file_infos.append(info)


print("\n" + "=" * 100)
print("SCANNING PROCESSED SEASON DIRECTORIES")
print("=" * 100)

for season, folder in PROCESSED_SEASON_DIRS.items():
    print(f"\n📅 {season} | {folder}")

    csvs = find_csvs(folder)
    print(f"CSV files found: {len(csvs)}")

    for path in csvs:
        info = scan_csv_file(path, season=season, source_group="processed_season")
        info["data_type_guess"] = classify_file(info["file_name"], info["file_path"])
        all_file_infos.append(info)


# Final combined CSV
if os.path.exists(FINAL_CSV):
    info = scan_csv_file(FINAL_CSV, season="all", source_group="final_combined")
    info["data_type_guess"] = "final_combined"
    all_file_infos.append(info)


# ============================================================
# 3. BUILD FILE INVENTORY TABLE
# ============================================================

inventory = pd.DataFrame(all_file_infos)

print("\n" + "=" * 100)
print("CSV INVENTORY SUMMARY")
print("=" * 100)

print("Total CSV files found:", len(inventory))

display(
    inventory[
        [
            "season",
            "source_group",
            "data_type_guess",
            "file_name",
            "n_rows",
            "n_cols",
            "size_kb",
            "file_path",
            "read_error",
        ]
    ].sort_values(["season", "source_group", "data_type_guess", "file_name"])
)


# ============================================================
# 4. SUMMARY BY DATA TYPE
# ============================================================

print("\n" + "=" * 100)
print("SUMMARY BY DATA TYPE")
print("=" * 100)

display(
    inventory
    .groupby(["source_group", "data_type_guess"], dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_size_kb=("size_kb", "sum"),
        total_rows=("n_rows", "sum"),
    )
    .reset_index()
    .sort_values(["source_group", "data_type_guess"])
)


# ============================================================
# 5. SHOW CANDIDATE PLAYER PERFORMANCE FILES
# ============================================================

print("\n" + "=" * 100)
print("CANDIDATE PLAYER PERFORMANCE FILES")
print("=" * 100)

player_files = inventory[
    inventory["data_type_guess"].isin(["player_performance", "sofascore", "fbref"])
].copy()

display(
    player_files[
        [
            "season",
            "source_group",
            "data_type_guess",
            "file_name",
            "n_rows",
            "n_cols",
            "file_path",
        ]
    ].sort_values(["season", "source_group", "file_name"])
)

print("\nColumns in candidate player performance files:")

for _, row in player_files.iterrows():
    print("\n" + "-" * 100)
    print(row["file_path"])
    print("Rows:", row["n_rows"], "| Columns:", row["n_cols"])
    print(row["columns"])


# ============================================================
# 6. SHOW CANDIDATE INJURY FILES
# ============================================================

print("\n" + "=" * 100)
print("CANDIDATE INJURY FILES")
print("=" * 100)

injury_files = inventory[inventory["data_type_guess"] == "injuries"].copy()

display(
    injury_files[
        [
            "season",
            "source_group",
            "data_type_guess",
            "file_name",
            "n_rows",
            "n_cols",
            "file_path",
        ]
    ].sort_values(["season", "source_group", "file_name"])
)

print("\nColumns in candidate injury files:")

for _, row in injury_files.iterrows():
    print("\n" + "-" * 100)
    print(row["file_path"])
    print("Rows:", row["n_rows"], "| Columns:", row["n_cols"])
    print(row["columns"])


# ============================================================
# 7. SHOW CANDIDATE FIXTURE / MATCH FILES
# ============================================================

print("\n" + "=" * 100)
print("CANDIDATE FIXTURE / MATCH FILES")
print("=" * 100)

fixture_files = inventory[inventory["data_type_guess"].isin(["fixtures", "final_combined"])].copy()

display(
    fixture_files[
        [
            "season",
            "source_group",
            "data_type_guess",
            "file_name",
            "n_rows",
            "n_cols",
            "file_path",
        ]
    ].sort_values(["season", "source_group", "file_name"])
)

print("\nColumns in candidate fixture files:")

for _, row in fixture_files.iterrows():
    print("\n" + "-" * 100)
    print(row["file_path"])
    print("Rows:", row["n_rows"], "| Columns:", row["n_cols"])
    print(row["columns"])


# ============================================================
# 8. FINAL COMBINED CSV QUICK CHECK
# ============================================================

print("\n" + "=" * 100)
print("FINAL COMBINED CSV CHECK")
print("=" * 100)

if os.path.exists(FINAL_CSV):
    df_final = pd.read_csv(FINAL_CSV, nrows=1000)

    print("Final CSV path:", FINAL_CSV)
    print("Preview shape:", df_final.shape)
    print("Columns:")
    print(list(df_final.columns))

    print("\nFirst rows:")
    display(df_final.head())

    if "season" in df_final.columns:
        print("\nSeason values in first 1000 rows:")
        display(df_final["season"].value_counts(dropna=False))

    if "fixture_id" in df_final.columns:
        print("\nUnique fixture IDs in first 1000 rows:")
        print(df_final["fixture_id"].nunique())

    if "player_name" in df_final.columns:
        print("\nUnique players in first 1000 rows:")
        print(df_final["player_name"].nunique())

else:
    print("Final CSV not found:", FINAL_CSV)


CURRENT WORKING DIRECTORY
/home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data

Files and folders in current directory:
📄 01_exploratory_data_analysis.ipynb
📁 2022-2023/
📁 2023-2024/
📁 2024-2025/
📁 API_SEASON_2022_2023/
📁 API_SEASON_2023_2024/
📁 API_SEASON_2024_2025/
📄 Fixture_IQ_Data_Seasons_2022-2025.csv
📁 SEASON_2022_2023/
📁 SEASON_2023_2024/
📁 SEASON_2024_2025/

SCANNING RAW SEASON DIRECTORIES

📅 2022-2023 | ./2022-2023
CSV files found: 41

📅 2023-2024 | ./2023-2024
CSV files found: 662

📅 2024-2025 | ./2024-2025
CSV files found: 277

SCANNING API SEASON DIRECTORIES

📅 2022-2023 | ./API_SEASON_2022_2023
CSV files found: 12

📅 2023-2024 | ./API_SEASON_2023_2024
CSV files found: 12

📅 2024-2025 | ./API_SEASON_2024_2025
CSV files found: 12

SCANNING PROCESSED SEASON DIRECTORIES

📅 2022-2023 | ./SEASON_2022_2023
CSV files found: 715

📅 2023-2024 | ./SEASON_2023_2024
CSV files found: 700

📅 2024-2025 | ./SEASON_2024_2025
CSV files found: 745

CSV INVENTORY SUMMARY
Total CSV file

,season,source_group,data_type_guess,file_name,n_rows,n_cols,size_kb,file_path,read_error
980,2022-2023,api_season,player_performance,multi_competition_player_stats_2022_2023.csv,23455,34,4184.9,./API_SEASON_2022_2023/multi_competition_player_stats_2022_2023.csv,None
982,2022-2023,api_season,player_performance,player_stats_champions_league_2022_2023.csv,1710,34,307.0,./API_SEASON_2022_2023/player_stats_champions_league_2022_2023.csv,None
983,2022-2023,api_season,player_performance,player_stats_community_shield_2022_2023.csv,40,34,7.4,./API_SEASON_2022_2023/player_stats_community_shield_2022_2023.csv,None
984,2022-2023,api_season,player_performance,player_stats_fa_cup_2022_2023.csv,2876,34,483.1,./API_SEASON_2022_2023/player_stats_fa_cup_2022_2023.csv,None
985,2022-2023,api_season,player_performance,player_stats_league_cup_2022_2023.csv,3645,34,623.8,./API_SEASON_2022_2023/player_stats_league_cup_2022_2023.csv,None
...,...,...,...,...,...,...,...,...,...
972,2024-2025,raw_season,sofascore,premier_league_2024_2025_dm_cdm.csv,562,12,25.5,./2024-2025/sofascore/premier_league/position_profiles/premier_league_2024_2...,None
973,2024-2025,raw_season,sofascore,premier_league_2024_2025_fw_st.csv,562,11,22.8,./2024-2025/sofascore/premier_league/position_profiles/premier_league_2024_2...,None
974,2024-2025,raw_season,sofascore,premier_league_2024_2025_gk.csv,562,6,16.8,./2024-2025/sofascore/premier_league/position_profiles/premier_league_2024_2...,None
975,2024-2025,raw_season,sofascore,premier_league_2024_2025_wg_am.csv,562,12,24.0,./2024-2025/sofascore/premier_league/position_profiles/premier_league_2024_2...,None



SUMMARY BY DATA TYPE


,source_group,data_type_guess,n_files,total_size_kb,total_rows
0,api_season,player_performance,18,24699.4,137444
1,api_season,team_data,18,1788.9,10132
2,final_combined,final_combined,1,23380.7,68722
3,processed_season,fixtures,1289,3631.7,15083
4,processed_season,injuries,75,1870.4,23808
5,processed_season,other,66,170.6,979
6,processed_season,player_performance,718,3024.9,11714
7,processed_season,team_data,12,10.5,326
8,raw_season,fbref,12,5.2,326
9,raw_season,fixtures,563,33869.1,75965



CANDIDATE PLAYER PERFORMANCE FILES


...


Columns in candidate fixture files:

----------------------------------------------------------------------------------------------------
./2022-2023/fbref/chelsea_2022_2023/chelsea_2022_2023_matches_all.csv
Rows: 50 | Columns: 19
['Date', 'start_time', 'Competition', 'Round', 'Day', 'Venue', 'Result', 'goals_for', 'goals_against', 'Opponent', 'possession', 'attendance', 'captain', 'formation', 'opp_formation', 'referee', 'Match_Report_URL', 'Match_Report', 'notes']

----------------------------------------------------------------------------------------------------
./2022-2023/fbref/chelsea_2022_2023/match_reports/master_goalkeeper_stats.csv
Rows: 51 | Columns: 24
['season', 'tracked_team', 'match_date', 'competition', 'opponent_team', 'original_file', 'Match_ID', 'Match_Report_URL', 'Date', 'Competition', 'Round', 'Venue', 'Opponent', 'Result', 'Match_Report_Name', 'Team', 'Player', 'Nation', 'Age', 'Min', 'SoTA', 'GA', 'Saves', 'Save%']

--------------------------------------------

,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,api_player_position_raw,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red,season_label_x,is_home,opponent_team,goals_for,goals_against,result,points,team_shots_on_goal,team_total_shots,team_possession,team_corner_kicks,team_fouls,team_gk_saves,opp_shots_on_goal,opp_total_shots,opp_possession,sofascore_position_raw,ss_minutes,sofascore_rating,rest_days,high_congestion_flag,min_last_7d,acwr_ratio,consecutive_away_games,season_label_y,fbref_position_raw,fb_min,fb_goals,fb_assists,fb_shots,fb_sot,fb_tackles_won,fb_crosses,fb_interceptions,fb_fouls,fb_fouled,fb_offsides,squad_injured_count,squad_soft_tissue_count,squad_avg_days_out,is_injured,injury_days_out,injury_type,position_raw_selected,position_source_used,position_group,position_informative_raw,position_informative,is_multirole_player,next_sofascore_rating,rating_decline_flag,min_last_28d,next_api_rating,api_rating_decline_flag,next_is_substitute,next_minutes_played
0,946877,2022-09-14,Champions League,2022,Group G - 2,Manchester City,Borussia Dortmund,Borussia Dortmund,4,Luca Unbehaun,38,G,0,0.0,False,True,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,2022-2023,False,Manchester City,1,2,Loss,0,2.0,5.0,34.0,2.0,7.0,1.0,NaN,NaN,NaN,G,0.0,NaN,NaN,0.0,0.0,0.0,1.0,2022-2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,G,API-Football,Goalkeeper,G,Goalkeeper,0,NaN,0,0.0,NaN,0,True,0.0
1,946918,2022-10-25,Champions League,2022,Group G - 5,Borussia Dortmund,Manchester City,Borussia Dortmund,4,Luca Unbehaun,38,G,0,0.0,False,True,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,2022-2023,True,Manchester City,0,0,Draw,1,4.0,11.0,27.0,0.0,5.0,3.0,NaN,NaN,NaN,G,0.0,NaN,21.0,0.0,0.0,0.0,0.0,2022-2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,G,API-Football,Goalkeeper,G,Goalkeeper,0,NaN,0,0.0,NaN,0,True,0.0
2,971800,2023-03-07,Champions League,2022,Round of 16,Chelsea,Borussia Dortmund,Borussia Dortmund,4,Luca Unbehaun,38,G,0,0.0,False,True,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,2022-2023,False,Chelsea,0,2,Loss,0,4.0,13.0,60.0,3.0,9.0,2.0,NaN,NaN,NaN,G,0.0,NaN,21.0,0.0,0.0,0.0,1.0,2022-2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,G,API-Football,Goalkeeper,G,Goalkeeper,0,NaN,0,0.0,NaN,0,NaN,NaN
3,946854,2022-09-06,Champions League,2022,Group G - 1,Sevilla,Manchester City,Manchester City,5,Manuel Akanji,25,D,90,7.2,False,False,0,0,0,0,79,2,71.0,0,0,1,0,1,6,3,1,1,0,0,2022-2023,False,Sevilla,4,0,Win,3,10.0,24.0,62.0,7.0,8.0,1.0,NaN,NaN,NaN,D,90.0,7.1,NaN,0.0,0.0,0.0,1.0,2022-2023,CB,90.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,0,NaN,NaN,D,API-Football,Defender,CB,Defender,0,7.2,0,0.0,7.3,0,False,90.0
4,946877,2022-09-14,Champions League,2022,Group G - 2,Manchester City,Borussia Dortmund,Manchester City,5,Manuel Akanji,25,D,90,7.3,False,False,0,0,0,0,90,0,87.0,0,0,4,0,0,10,7,2,0,0,0,2022-2023,True,Borussia Dortmund,2,1,Win,3,3.0,12.0,66.0,6.0,9.0,1.0,NaN,NaN,NaN,D,90.0,7.2,8.0,0.0,0.0,0.0,0.0,2022-2023,CB,90.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,2.0,0.0,NaN,NaN,NaN,0,NaN,NaN,D,API-Football,Defender,CB,Defender,0,7.4,0,90.0,7.5,0,False,90.0



Season values in first 1000 rows:


season
2022    417
2023    379
2024    204
Name: count, dtype: int64


Unique fixture IDs in first 1000 rows:
650

Unique players in first 1000 rows:
34

SCAN COMPLETE - NOTHING SAVED

Next step:
1. Use the CANDIDATE PLAYER PERFORMANCE FILES section to decide which file gives player performance per match.
2. Use the CANDIDATE INJURY FILES section to decide which injury table should be merged.
3. Use the CANDIDATE FIXTURE / MATCH FILES section to decide the match-level base table.
4. After that, analyse each source independently before merging.

No CSV was created or modified by this cell.



## Section 2 — Data Source A: API Multi-Competition Player Stats
Per-match player stats across **all competitions** (Premier League, Champions League, FA Cup, League Cup).
This is the **widest coverage** dataset — all 20 PL teams, all competitions, 3 seasons.


In [9]:
# Section 2 — API Multi-Competition Player Stats
# Main source for player performance per match

import os
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

api_player_dfs = {}

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"API_SEASON_{season_key}",
        f"multi_competition_player_stats_{season_key}.csv"
    )

    print("\n" + "=" * 80)
    print(season)
    print(path)

    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["date"])
        api_player_dfs[season] = df

        print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print("Columns:")
        print(list(df.columns))

        print("\nCompetitions:")
        print(df["competition"].value_counts())

        print("\nSample:")
        display(df.head(3))

    else:
        print("File not found")

# Combine all seasons in memory
api_players_all = pd.concat(api_player_dfs.values(), ignore_index=True)

print("\n" + "=" * 80)
print("COMBINED API PLAYER DATA")
print("=" * 80)

print("Shape:", api_players_all.shape)

print("\nRows by season:")
print(api_players_all["season"].value_counts().sort_index())

print("\nUnique fixtures:", api_players_all["fixture_id"].nunique())
print("Unique players:", api_players_all["player_id"].nunique())

display(api_players_all.head())


2022-2023
./API_SEASON_2022_2023/multi_competition_player_stats_2022_2023.csv
Loaded: 23,455 rows × 34 columns
Columns:
['fixture_id', 'date', 'competition', 'season', 'round', 'home_team', 'away_team', 'player_team', 'player_id', 'player_name', 'player_number', 'player_position', 'minutes_played', 'rating', 'is_captain', 'is_substitute', 'shots_total', 'shots_on_target', 'goals', 'assists', 'passes_total', 'passes_key', 'passes_accuracy', 'dribbles_attempts', 'dribbles_success', 'tackles_total', 'tackles_blocks', 'tackles_interceptions', 'duels_total', 'duels_won', 'fouls_drawn', 'fouls_committed', 'cards_yellow', 'cards_red']

Competitions:
competition
Premier League      15184
League Cup           3645
FA Cup               2876
Champions League     1710
Community Shield       40
Name: count, dtype: int64

Sample:


,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,player_position,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red
0,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18835,Vicente Guaita,13,G,90,6.2,False,False,0,0,0,0,37,0,35.0,0,0,0,0,0,0,0,0,0,0,0
1,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18862,Nathaniel Clyne,17,D,90,6.3,False,False,0,0,0,0,56,1,50.0,1,1,0,0,1,5,3,1,1,1,0
2,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,2729,Joachim Andersen,16,D,90,7.9,False,False,0,0,0,0,104,1,91.0,0,0,6,0,2,15,12,2,1,0,0



2023-2024
./API_SEASON_2023_2024/multi_competition_player_stats_2023_2024.csv
Loaded: 22,021 rows × 34 columns
Columns:
['fixture_id', 'date', 'competition', 'season', 'round', 'home_team', 'away_team', 'player_team', 'player_id', 'player_name', 'player_number', 'player_position', 'minutes_played', 'rating', 'is_captain', 'is_substitute', 'shots_total', 'shots_on_target', 'goals', 'assists', 'passes_total', 'passes_key', 'passes_accuracy', 'dribbles_attempts', 'dribbles_success', 'tackles_total', 'tackles_blocks', 'tackles_interceptions', 'duels_total', 'duels_won', 'fouls_drawn', 'fouls_committed', 'cards_yellow', 'cards_red']

Competitions:
competition
Premier League      15180
League Cup           3669
FA Cup               1756
Champions League     1376
Community Shield       40
Name: count, dtype: int64

Sample:


,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,player_position,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red
0,1035037,2023-08-11 19:00:00+00:00,Premier League,2023,Regular Season - 1,Burnley,Manchester City,Burnley,162489,James Trafford,1,G,90,7.5,False,False,0,0,0,0,44,0,31.0,0,0,0,0,0,0,0,0,0,0,0
1,1035037,2023-08-11 19:00:00+00:00,Premier League,2023,Regular Season - 1,Burnley,Manchester City,Burnley,19331,Connor Roberts,14,D,90,6.5,False,False,0,0,0,0,38,0,33.0,0,0,2,0,0,4,3,0,1,0,0
2,1035037,2023-08-11 19:00:00+00:00,Premier League,2023,Regular Season - 1,Burnley,Manchester City,Burnley,323449,Ameen Al-Dakhil,28,D,90,6.3,False,False,0,0,0,0,36,0,30.0,0,0,2,1,1,11,5,1,1,0,0



2024-2025
./API_SEASON_2024_2025/multi_competition_player_stats_2024_2025.csv
Loaded: 23,246 rows × 34 columns
Columns:
['fixture_id', 'date', 'competition', 'season', 'round', 'home_team', 'away_team', 'player_team', 'player_id', 'player_name', 'player_number', 'player_position', 'minutes_played', 'rating', 'is_captain', 'is_substitute', 'shots_total', 'shots_on_target', 'goals', 'assists', 'passes_total', 'passes_key', 'passes_accuracy', 'dribbles_attempts', 'dribbles_success', 'tackles_total', 'tackles_blocks', 'tackles_interceptions', 'duels_total', 'duels_won', 'fouls_drawn', 'fouls_committed', 'cards_yellow', 'cards_red']

Competitions:
competition
Premier League      15188
League Cup           3683
FA Cup               2340
Champions League     1995
Community Shield       40
Name: count, dtype: int64

Sample:


,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,player_position,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red
0,1208021,2024-08-16 19:00:00+00:00,Premier League,2024,Regular Season - 1,Manchester United,Fulham,Manchester United,526,André Onana,24,G,90,7.2,False,False,0,0,0,0,23,0,16.0,0,0,0,0,0,0,0,0,0,0,0
1,1208021,2024-08-16 19:00:00+00:00,Premier League,2024,Regular Season - 1,Manchester United,Fulham,Manchester United,545,Noussair Mazraoui,3,D,81,7.5,False,False,0,0,0,0,38,0,35.0,0,0,2,0,3,7,6,1,0,0,0
2,1208021,2024-08-16 19:00:00+00:00,Premier League,2024,Regular Season - 1,Manchester United,Fulham,Manchester United,2935,Harry Maguire,5,D,81,7.3,False,False,0,0,0,0,60,0,50.0,2,1,0,1,3,9,6,0,2,1,0



COMBINED API PLAYER DATA
Shape: (68722, 34)

Rows by season:
season
2022    23455
2023    22021
2024    23246
Name: count, dtype: int64

Unique fixtures: 1715
Unique players: 5289


,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,player_position,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red
0,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18835,Vicente Guaita,13,G,90,6.2,False,False,0,0,0,0,37,0,35.0,0,0,0,0,0,0,0,0,0,0,0
1,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18862,Nathaniel Clyne,17,D,90,6.3,False,False,0,0,0,0,56,1,50.0,1,1,0,0,1,5,3,1,1,1,0
2,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,2729,Joachim Andersen,16,D,90,7.9,False,False,0,0,0,0,104,1,91.0,0,0,6,0,2,15,12,2,1,0,0
3,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,67971,Marc Guéhi,6,D,90,6.3,True,False,0,0,0,0,105,0,93.0,0,0,0,2,1,7,1,0,1,0,0
4,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,182201,Tyrick Mitchell,3,D,90,6.3,False,False,0,0,0,0,52,0,39.0,1,0,1,1,2,7,2,1,2,0,0


## Section 3 — Data Source B: API Team Results (with Match Stats)
Per-match **team-level** stats including possession, shots, xG for both home and away.
Used to add match context (was it a tough opponent? high-intensity game?) to each player row.


In [11]:
# Section 3 — API Team Results
# Team-level match stats: possession, shots, xG, fouls, cards
# Nothing is saved

import os
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

api_team_dfs = {}

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"API_SEASON_{season_key}",
        f"multi_competition_team_results_{season_key}.csv"
    )

    print("\n" + "=" * 80)
    print(season)
    print(path)

    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["date"])
        api_team_dfs[season] = df

        print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print("Columns:")
        print(list(df.columns))

        print("\nCompetitions:")
        print(df["competition"].value_counts())

        print("\nSample:")
        display(df.head(3))

    else:
        print("File not found")

# Combine all seasons in memory
api_teams_all = pd.concat(api_team_dfs.values(), ignore_index=True)

print("\n" + "=" * 80)
print("COMBINED API TEAM DATA")
print("=" * 80)

print("Shape:", api_teams_all.shape)

print("\nRows by season:")
print(api_teams_all["season"].value_counts().sort_index())

print("\nUnique fixtures:", api_teams_all["fixture_id"].nunique())

if "team_name" in api_teams_all.columns:
    print("Unique teams:", api_teams_all["team_name"].nunique())
elif "team" in api_teams_all.columns:
    print("Unique teams:", api_teams_all["team"].nunique())

display(api_teams_all.head())


2022-2023
./API_SEASON_2022_2023/multi_competition_team_results_2022_2023.csv
Loaded: 2,802 rows × 49 columns
Columns:
['fixture_id', 'date', 'competition', 'season', 'round', 'team_name', 'team_id', 'is_home', 'opponent', 'opponent_id', 'goals_for', 'goals_against', 'result', 'points', 'status', 'home_shots_on_goal', 'home_shots_off_goal', 'home_total_shots', 'home_blocked_shots', 'home_shots_insidebox', 'home_shots_outsidebox', 'home_fouls', 'home_corner_kicks', 'home_offsides', 'home_ball_possession', 'home_yellow_cards', 'home_goalkeeper_saves', 'home_total_passes', 'home_passes_accurate', 'home_passes_%', 'away_shots_on_goal', 'away_shots_off_goal', 'away_total_shots', 'away_blocked_shots', 'away_shots_insidebox', 'away_shots_outsidebox', 'away_fouls', 'away_corner_kicks', 'away_offsides', 'away_ball_possession', 'away_yellow_cards', 'away_goalkeeper_saves', 'away_total_passes', 'away_passes_accurate', 'away_passes_%', 'home_red_cards', 'away_red_cards', 'home_expected_goals', 'a

,fixture_id,date,competition,season,round,team_name,team_id,is_home,opponent,opponent_id,goals_for,goals_against,result,points,status,home_shots_on_goal,home_shots_off_goal,home_total_shots,home_blocked_shots,home_shots_insidebox,home_shots_outsidebox,home_fouls,home_corner_kicks,home_offsides,home_ball_possession,home_yellow_cards,home_goalkeeper_saves,home_total_passes,home_passes_accurate,home_passes_%,away_shots_on_goal,away_shots_off_goal,away_total_shots,away_blocked_shots,away_shots_insidebox,away_shots_outsidebox,away_fouls,away_corner_kicks,away_offsides,away_ball_possession,away_yellow_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,home_red_cards,away_red_cards,home_expected_goals,away_expected_goals
0,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,52,True,Arsenal,42,0,2,Loss,0,FT,2.0,2.0,10.0,6.0,9.0,1.0,16.0,3.0,1.0,56%,1.0,1.0,562.0,487.0,87%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Arsenal,42,False,Crystal Palace,52,2,0,Win,3,FT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,10.0,4.0,8.0,2.0,11.0,5.0,2.0,44%,2.0,2.0,438.0,360.0,82%,NaN,NaN,NaN,NaN
2,867947,2022-08-06 11:30:00+00:00,Premier League,2022,Regular Season - 1,Fulham,36,True,Liverpool,40,2,2,Draw,1,FT,3.0,2.0,9.0,4.0,7.0,2.0,7.0,4.0,4.0,33%,2.0,1.0,294.0,181.0,62%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



2023-2024
./API_SEASON_2023_2024/multi_competition_team_results_2023_2024.csv
Loaded: 1,106 rows × 51 columns
Columns:
['fixture_id', 'date', 'competition', 'season', 'round', 'team_name', 'team_id', 'is_home', 'opponent', 'opponent_id', 'goals_for', 'goals_against', 'result', 'points', 'status', 'home_shots_on_goal', 'home_shots_off_goal', 'home_total_shots', 'home_blocked_shots', 'home_shots_insidebox', 'home_shots_outsidebox', 'home_fouls', 'home_corner_kicks', 'home_offsides', 'home_ball_possession', 'home_red_cards', 'home_goalkeeper_saves', 'home_total_passes', 'home_passes_accurate', 'home_passes_%', 'home_expected_goals', 'away_shots_on_goal', 'away_shots_off_goal', 'away_total_shots', 'away_blocked_shots', 'away_shots_insidebox', 'away_shots_outsidebox', 'away_fouls', 'away_corner_kicks', 'away_offsides', 'away_ball_possession', 'away_red_cards', 'away_goalkeeper_saves', 'away_total_passes', 'away_passes_accurate', 'away_passes_%', 'away_expected_goals', 'home_yellow_cards', 

,fixture_id,date,competition,season,round,team_name,team_id,is_home,opponent,opponent_id,goals_for,goals_against,result,points,status,home_shots_on_goal,home_shots_off_goal,home_total_shots,home_blocked_shots,home_shots_insidebox,home_shots_outsidebox,home_fouls,home_corner_kicks,home_offsides,home_ball_possession,home_red_cards,home_goalkeeper_saves,home_total_passes,home_passes_accurate,home_passes_%,home_expected_goals,away_shots_on_goal,away_shots_off_goal,away_total_shots,away_blocked_shots,away_shots_insidebox,away_shots_outsidebox,away_fouls,away_corner_kicks,away_offsides,away_ball_possession,away_red_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,away_expected_goals,home_yellow_cards,away_yellow_cards,home_goals_prevented,away_goals_prevented
0,1035037,2023-08-11 19:00:00+00:00,Premier League,2023,Regular Season - 1,Burnley,44,True,Manchester City,50,0,3,Loss,0,FT,1.0,3.0,6.0,2.0,5.0,1.0,11.0,6.0,0.0,34%,1.0,5.0,365.0,290.0,79%,0.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1035037,2023-08-11 19:00:00+00:00,Premier League,2023,Regular Season - 1,Manchester City,50,False,Burnley,44,3,0,Win,3,FT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,4.0,17.0,5.0,14.0,3.0,8.0,5.0,1.0,66%,0.0,1.0,706.0,634.0,90%,2.08,NaN,NaN,NaN,NaN
2,1035038,2023-08-12 11:30:00+00:00,Premier League,2023,Regular Season - 1,Arsenal,42,True,Nottingham Forest,65,2,1,Win,3,FT,7.0,3.0,15.0,5.0,8.0,7.0,12.0,8.0,2.0,78%,NaN,1.0,769.0,693.0,90%,0.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN



2024-2025
./API_SEASON_2024_2025/multi_competition_team_results_2024_2025.csv
Loaded: 1,158 rows × 53 columns
Columns:
['fixture_id', 'team_name', 'is_home', 'date', 'competition', 'season', 'round', 'team_id', 'opponent', 'opponent_id', 'goals_for', 'goals_against', 'result', 'points', 'status', 'home_team', 'away_team', 'home_shots_on_goal', 'home_shots_off_goal', 'home_total_shots', 'home_blocked_shots', 'home_shots_insidebox', 'home_shots_outsidebox', 'home_fouls', 'home_corner_kicks', 'home_offsides', 'home_ball_possession', 'home_yellow_cards', 'home_goalkeeper_saves', 'home_total_passes', 'home_passes_accurate', 'home_passes_%', 'away_shots_on_goal', 'away_shots_off_goal', 'away_total_shots', 'away_blocked_shots', 'away_shots_insidebox', 'away_shots_outsidebox', 'away_fouls', 'away_corner_kicks', 'away_offsides', 'away_ball_possession', 'away_yellow_cards', 'away_goalkeeper_saves', 'away_total_passes', 'away_passes_accurate', 'away_passes_%', 'home_red_cards', 'away_red_cards',

,fixture_id,team_name,is_home,date,competition,season,round,team_id,opponent,opponent_id,goals_for,goals_against,result,points,status,home_team,away_team,home_shots_on_goal,home_shots_off_goal,home_total_shots,home_blocked_shots,home_shots_insidebox,home_shots_outsidebox,home_fouls,home_corner_kicks,home_offsides,home_ball_possession,home_yellow_cards,home_goalkeeper_saves,home_total_passes,home_passes_accurate,home_passes_%,away_shots_on_goal,away_shots_off_goal,away_total_shots,away_blocked_shots,away_shots_insidebox,away_shots_outsidebox,away_fouls,away_corner_kicks,away_offsides,away_ball_possession,away_yellow_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,home_red_cards,away_red_cards,home_expected_goals,home_goals_prevented,away_expected_goals,away_goals_prevented
0,1208021,Fulham,False,2024-08-16,Premier League,2024,Regular Season - 1,NaN,Manchester United,NaN,0,1,Loss,0,NaN,Manchester United,Fulham,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1208021,Manchester United,True,2024-08-16,Premier League,2024,Regular Season - 1,NaN,Fulham,NaN,1,0,Win,3,NaN,Manchester United,Fulham,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1208022,Liverpool,False,2024-08-17,Premier League,2024,Regular Season - 1,NaN,Ipswich,NaN,2,0,Win,3,NaN,Ipswich,Liverpool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



COMBINED API TEAM DATA
Shape: (5066, 53)

Rows by season:
season
2022    2802
2023    1106
2024    1158
Name: count, dtype: int64

Unique fixtures: 2533
Unique teams: 768


,fixture_id,date,competition,season,round,team_name,team_id,is_home,opponent,opponent_id,goals_for,goals_against,result,points,status,home_shots_on_goal,home_shots_off_goal,home_total_shots,home_blocked_shots,home_shots_insidebox,home_shots_outsidebox,home_fouls,home_corner_kicks,home_offsides,home_ball_possession,home_yellow_cards,home_goalkeeper_saves,home_total_passes,home_passes_accurate,home_passes_%,away_shots_on_goal,away_shots_off_goal,away_total_shots,away_blocked_shots,away_shots_insidebox,away_shots_outsidebox,away_fouls,away_corner_kicks,away_offsides,away_ball_possession,away_yellow_cards,away_goalkeeper_saves,away_total_passes,away_passes_accurate,away_passes_%,home_red_cards,away_red_cards,home_expected_goals,away_expected_goals,home_goals_prevented,away_goals_prevented,home_team,away_team
0,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,52.0,True,Arsenal,42.0,0,2,Loss,0,FT,2.0,2.0,10.0,6.0,9.0,1.0,16.0,3.0,1.0,56%,1.0,1.0,562.0,487.0,87%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Arsenal,42.0,False,Crystal Palace,52.0,2,0,Win,3,FT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,10.0,4.0,8.0,2.0,11.0,5.0,2.0,44%,2.0,2.0,438.0,360.0,82%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,867947,2022-08-06 11:30:00+00:00,Premier League,2022,Regular Season - 1,Fulham,36.0,True,Liverpool,40.0,2,2,Draw,1,FT,3.0,2.0,9.0,4.0,7.0,2.0,7.0,4.0,4.0,33%,2.0,1.0,294.0,181.0,62%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,867947,2022-08-06 11:30:00+00:00,Premier League,2022,Regular Season - 1,Liverpool,40.0,False,Fulham,36.0,2,2,Draw,1,FT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5.0,11.0,2.0,9.0,2.0,9.0,4.0,4.0,67%,0.0,1.0,612.0,473.0,77%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,867951,2022-08-06 14:00:00+00:00,Premier League,2022,Regular Season - 1,Newcastle,34.0,True,Nottingham Forest,65.0,2,0,Win,3,FT,10.0,8.0,23.0,5.0,16.0,7.0,9.0,11.0,2.0,61%,0.0,0.0,469.0,379.0,81%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Section 4 — Data Source C: SofaScore Dynamic Analytics
Per-match **workload & fatigue metrics** calculated dynamically for every player.
Covers all 20 PL teams. **Only available for Premier League fixtures.**
Key fields: `rest_days`, `high_congestion_flag`, `min_last_7d`, `acwr_ratio`, `consecutive_away_games`.


In [12]:
# Section 4 — SofaScore Dynamic Analytics
# Workload/fatigue metrics per player-match
# Premier League only
# Nothing is saved

import os
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

sofascore_dynamic = {}

for season in SEASONS:
    path = os.path.join(
        BASE_DIR,
        season,
        "sofascore_dynamic",
        "fixtureiq_dynamic_analytics_clean.csv"
    )

    print("\n" + "=" * 80)
    print(season)
    print(path)

    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["match_date_str"])
        sofascore_dynamic[season] = df

        print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print("Columns:")
        print(list(df.columns))

        print("\nSample:")
        display(df.head(3))

    else:
        print("File not found")

# Combine all seasons in memory
sofascore_dynamic_all = pd.concat(sofascore_dynamic.values(), ignore_index=True)

print("\n" + "=" * 80)
print("COMBINED SOFASCORE DYNAMIC DATA")
print("=" * 80)

print("Shape:", sofascore_dynamic_all.shape)

print("\nRows by season:")
for season, df in sofascore_dynamic.items():
    print(season, ":", len(df))

print("\nKey metric summary:")
key_cols = [
    "rest_days",
    "high_congestion_flag",
    "min_last_7d",
    "acwr_ratio",
    "consecutive_away_games",
    "minutesPlayed",
    "rating",
]

available_key_cols = [c for c in key_cols if c in sofascore_dynamic_all.columns]

display(sofascore_dynamic_all[available_key_cols].describe())

print("\nNulls in key metrics:")
display(sofascore_dynamic_all[available_key_cols].isna().sum())

display(sofascore_dynamic_all.head())


2022-2023
./2022-2023/sofascore_dynamic/fixtureiq_dynamic_analytics_clean.csv
Loaded: 6,500 rows × 19 columns
Columns:
['match_date_str', 'match_id', 'competition', 'teamName', 'name', 'position', 'rating', 'elo', 'is_away', 'consecutive_away_games', 'minutesPlayed', 'rest_days', 'high_congestion_flag', 'min_last_7d', 'acwr_ratio', 'lineup_churn', 'team_xg', 'team_xga', 'xg_difference']

Sample:


,match_date_str,match_id,competition,teamName,name,position,rating,elo,is_away,consecutive_away_games,minutesPlayed,rest_days,high_congestion_flag,min_last_7d,acwr_ratio,lineup_churn,team_xg,team_xga,xg_difference
0,2022-08-07,10385514,England Premier League,West Ham United,Aaron Cresswell,D,6.7,1751.339966,0,0,90,14,0,0,0.0,0,NaN,NaN,NaN
1,2022-08-31,10385564,England Premier League,West Ham United,Aaron Cresswell,D,7.1,1751.339966,0,0,72,24,0,0,0.0,4,NaN,NaN,NaN
2,2022-10-19,10385729,England Premier League,West Ham United,Aaron Cresswell,D,7.8,1751.339966,1,1,90,49,0,0,0.0,4,NaN,NaN,NaN



2023-2024
./2023-2024/sofascore_dynamic/fixtureiq_dynamic_analytics_clean.csv
Loaded: 13,142 rows × 17 columns
Columns:
['match_date_str', 'match_id', 'competition', 'teamName', 'name', 'position', 'rating', 'match_type', 'cohort_group', 'is_away', 'consecutive_away_games', 'minutesPlayed', 'rest_days', 'high_congestion_flag', 'min_last_7d', 'acwr_ratio', 'lineup_churn']

Sample:


,match_date_str,match_id,competition,teamName,name,position,rating,match_type,cohort_group,is_away,consecutive_away_games,minutesPlayed,rest_days,high_congestion_flag,min_last_7d,acwr_ratio,lineup_churn
0,2023-08-20,11352418,England Premier League,West Ham United,Aaron Cresswell,D,NaN,Domestic,Baseline,0,0,0,14,0,0,0.0,3
1,2023-08-26,11352466,England Premier League,West Ham United,Aaron Cresswell,D,NaN,Domestic,Baseline,1,1,0,6,0,0,0.0,2
2,2023-09-01,11352590,England Premier League,West Ham United,Aaron Cresswell,D,NaN,Domestic,Baseline,1,2,0,6,0,0,0.0,2



2024-2025
./2024-2025/sofascore_dynamic/fixtureiq_dynamic_analytics_clean.csv
Loaded: 6,726 rows × 19 columns
Columns:
['match_date_str', 'match_id', 'competition', 'teamName', 'name', 'position', 'rating', 'elo', 'is_away', 'consecutive_away_games', 'minutesPlayed', 'rest_days', 'high_congestion_flag', 'min_last_7d', 'acwr_ratio', 'lineup_churn', 'team_xg', 'team_xga', 'xg_difference']

Sample:


...


COMBINED SOFASCORE DYNAMIC DATA
Shape: (26368, 21)

Rows by season:
2022-2023 : 6500
2023-2024 : 13142
2024-2025 : 6726

Key metric summary:


,rest_days,high_congestion_flag,min_last_7d,acwr_ratio,consecutive_away_games,minutesPlayed,rating
count,26368.000000,26368.000000,26368.000000,26368.000000,26368.000000,26368.000000,20030.000000
mean,15.048809,0.145593,31.473756,0.741275,0.663228,50.209117,6.914603
std,21.219176,0.352705,47.371104,1.142630,0.791243,38.525415,0.570091
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
25%,4.000000,0.000000,0.000000,0.000000,0.000000,8.000000,6.500000
50%,7.000000,0.000000,0.000000,0.000000,0.000000,64.000000,6.800000
75%,14.000000,0.000000,76.000000,1.322576,1.000000,90.000000,7.200000
max,294.000000,1.000000,180.000000,4.000000,5.000000,120.000000,10.000000



Nulls in key metrics:


rest_days                    0
high_congestion_flag         0
min_last_7d                  0
acwr_ratio                   0
consecutive_away_games       0
minutesPlayed                0
rating                    6338
dtype: int64

,match_date_str,match_id,competition,teamName,name,position,rating,elo,is_away,consecutive_away_games,minutesPlayed,rest_days,high_congestion_flag,min_last_7d,acwr_ratio,lineup_churn,team_xg,team_xga,xg_difference,match_type,cohort_group
0,2022-08-07,10385514,England Premier League,West Ham United,Aaron Cresswell,D,6.7,1751.339966,0,0,90,14,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN
1,2022-08-31,10385564,England Premier League,West Ham United,Aaron Cresswell,D,7.1,1751.339966,0,0,72,24,0,0,0.0,4,NaN,NaN,NaN,NaN,NaN
2,2022-10-19,10385729,England Premier League,West Ham United,Aaron Cresswell,D,7.8,1751.339966,1,1,90,49,0,0,0.0,4,NaN,NaN,NaN,NaN,NaN
3,2023-02-19,10385477,England Premier League,West Ham United,Aaron Cresswell,D,NaN,1751.339966,1,2,0,123,0,0,0.0,1,NaN,NaN,NaN,NaN,NaN
4,2023-04-26,10385616,England Premier League,West Ham United,Aaron Cresswell,D,6.3,1751.339966,0,0,90,66,0,0,0.0,4,NaN,NaN,NaN,NaN,NaN


In [13]:
# Section 4C2 — SofaScore Dynamic Master
# Full SofaScore per-match detail + workload variables
# Nothing is saved

import os
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

sofascore_master = {}

for season in SEASONS:
    path = os.path.join(
        BASE_DIR,
        season,
        "sofascore_dynamic",
        "fixtureiq_dynamic_master.csv"
    )

    print("\n" + "=" * 80)
    print(season)
    print(path)

    if os.path.exists(path):
        df = pd.read_csv(path)
        sofascore_master[season] = df

        print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print("Columns:")
        print(list(df.columns))

        print("\nSample:")
        display(df.head(2))

    else:
        print("File not found")

# Combine all seasons in memory
sofascore_master_all = pd.concat(sofascore_master.values(), ignore_index=True)

print("\n" + "=" * 80)
print("COMBINED SOFASCORE MASTER DATA")
print("=" * 80)

print("Shape:", sofascore_master_all.shape)

print("\nRows by season:")
for season, df in sofascore_master.items():
    print(season, ":", len(df))

print("\nUnique matches:")
if "match_id" in sofascore_master_all.columns:
    print(sofascore_master_all["match_id"].nunique())

print("\nUnique players:")
if "sofascoreId" in sofascore_master_all.columns:
    print(sofascore_master_all["sofascoreId"].nunique())
elif "id" in sofascore_master_all.columns:
    print(sofascore_master_all["id"].nunique())

print("\nImportant workload columns:")
workload_cols = [
    "rest_days",
    "high_congestion_flag",
    "min_last_7d",
    "min_last_28d",
    "acwr_ratio",
    "consecutive_away_games",
]

print([c for c in workload_cols if c in sofascore_master_all.columns])

print("\nImportant performance columns:")
performance_cols = [
    "rating",
    "minutesPlayed",
    "goals",
    "goalAssist",
    "totalShots",
    "onTargetScoringAttempt",
    "totalPass",
    "accuratePass",
    "keyPass",
    "totalTackle",
    "wonTackle",
    "interceptionWon",
    "duelWon",
    "duelLost",
]

print([c for c in performance_cols if c in sofascore_master_all.columns])

display(sofascore_master_all.head())



2022-2023
./2022-2023/sofascore_dynamic/fixtureiq_dynamic_master.csv
Loaded: 7,304 rows × 108 columns
Columns:
['date', 'name', 'slug', 'shortName', 'position', 'jerseyNumber', 'height', 'userCount', 'gender', 'country', 'id', 'marketValueCurrency', 'dateOfBirthTimestamp', 'proposedMarketValueRaw', 'fieldTranslations', 'firstName', 'lastName', 'sofascoreId', 'teamId', 'shirtNumber', 'substitute', 'totalPass', 'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'accurateOwnHalfPasses', 'totalOwnHalfPasses', 'accurateOppositionHalfPasses', 'totalOppositionHalfPasses', 'aerialWon', 'duelWon', 'ballRecovery', 'goodHighClaim', 'savedShotsFromInsideTheBox', 'saves', 'minutesPlayed', 'touches', 'rating', 'possessionLostCtrl', 'expectedGoals', 'expectedGoalsOnTarget', 'expectedAssists', 'totalShots', 'goalsPrevented', 'statisticsType', 'goalAssist', 'totalCross', 'accurateCross', 'aerialLost', 'duelLost', 'totalClearance', 'interceptionWon', 'totalTackle', 'wonTackle', 'wasFouled', 'fouls'

,date,name,slug,shortName,position,jerseyNumber,height,userCount,gender,country,id,marketValueCurrency,dateOfBirthTimestamp,proposedMarketValueRaw,fieldTranslations,firstName,lastName,sofascoreId,teamId,shirtNumber,substitute,totalPass,accuratePass,totalLongBalls,accurateLongBalls,accurateOwnHalfPasses,totalOwnHalfPasses,accurateOppositionHalfPasses,totalOppositionHalfPasses,aerialWon,duelWon,ballRecovery,goodHighClaim,savedShotsFromInsideTheBox,saves,minutesPlayed,touches,rating,possessionLostCtrl,expectedGoals,expectedGoalsOnTarget,expectedAssists,totalShots,goalsPrevented,statisticsType,goalAssist,totalCross,accurateCross,aerialLost,duelLost,totalClearance,interceptionWon,totalTackle,wonTackle,wasFouled,fouls,keyPass,challengeLost,totalContest,wonContest,clearanceOffLine,outfielderBlock,unsuccessfulTouch,blockedScoringAttempt,dispossessed,shotOffTarget,hitWoodwork,totalOffside,onTargetScoringAttempt,goals,penaltyWon,penaltyFaced,totalKeeperSweeper,accurateKeeperSweeper,penaltyConceded,bigChanceCreated,bigChanceMissed,teamName,captain,match_id,match_date_str,competition,home_team_name,away_team_name,lastManTackle,ownGoals,punches,crossNotClaimed,errorLeadToAGoal,errorLeadToAShot,penaltySave,penaltyMiss,ratingVersions,rest_days,high_congestion_flag,min_last_7d,min_last_28d,acwr_ratio,is_away,consecutive_away_games,lineup_churn,elo,home_team,home_xg,away_xg,team_xg,team_xga,xg_difference
0,2022-08-07,Aaron Cresswell,aaron-cresswell,A. Cresswell,D,3.0,170,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,NaN,NaN,37,3,False,22.0,17.0,2.0,1.0,12.0,15.0,7.0,10.0,1.0,2.0,5.0,NaN,NaN,NaN,90,35.0,6.7,6.0,0.0,0.0,0.103592,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,3.0,2.0,NaN,1.0,4.0,2.0,1.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,10385514,2022-08-07,England Premier League,West Ham United,Manchester City,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,0,0,0,0.0,0,0,0,1751.339966,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-31,Aaron Cresswell,aaron-cresswell,A. Cresswell,D,3.0,170,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,NaN,NaN,37,3,False,32.0,29.0,1.0,1.0,14.0,15.0,15.0,18.0,1.0,2.0,2.0,NaN,NaN,NaN,72,47.0,7.1,4.0,0.0,0.0,0.007961,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,1.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,10385564,2022-08-31,England Premier League,West Ham United,Tottenham Hotspur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,0,0,90,0.0,0,0,4,1751.339966,NaN,NaN,NaN,NaN,NaN,NaN



2023-2024
./2023-2024/sofascore_dynamic/fixtureiq_dynamic_master.csv
Loaded: 14,758 rows × 106 columns
Columns:
['date', 'name', 'firstName', 'lastName', 'slug', 'shortName', 'position', 'jerseyNumber', 'height', 'userCount', 'gender', 'country', 'id', 'marketValueCurrency', 'dateOfBirthTimestamp', 'proposedMarketValueRaw', 'fieldTranslations', 'sofascoreId', 'teamId', 'shirtNumber', 'substitute', 'totalPass', 'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'accurateOwnHalfPasses', 'totalOwnHalfPasses', 'accurateOppositionHalfPasses', 'totalOppositionHalfPasses', 'totalClearance', 'ballRecovery', 'savedShotsFromInsideTheBox', 'saves', 'totalKeeperSweeper', 'accurateKeeperSweeper', 'minutesPlayed', 'touches', 'rating', 'possessionLostCtrl', 'expectedGoals', 'expectedGoalsOnTarget', 'expectedAssists', 'ratingVersions', 'totalShots', 'goalsPrevented', 'statisticsType', 'aerialWon', 'duelLost', 'duelWon', 'totalTackle', 'wonTackle', 'unsuccessfulTouch', 'fouls', 'aerialLost', 'chal

,date,name,firstName,lastName,slug,shortName,position,jerseyNumber,height,userCount,gender,country,id,marketValueCurrency,dateOfBirthTimestamp,proposedMarketValueRaw,fieldTranslations,sofascoreId,teamId,shirtNumber,substitute,totalPass,accuratePass,totalLongBalls,accurateLongBalls,accurateOwnHalfPasses,totalOwnHalfPasses,accurateOppositionHalfPasses,totalOppositionHalfPasses,totalClearance,ballRecovery,savedShotsFromInsideTheBox,saves,totalKeeperSweeper,accurateKeeperSweeper,minutesPlayed,touches,rating,possessionLostCtrl,expectedGoals,expectedGoalsOnTarget,expectedAssists,ratingVersions,totalShots,goalsPrevented,statisticsType,aerialWon,duelLost,duelWon,totalTackle,wonTackle,unsuccessfulTouch,fouls,aerialLost,challengeLost,dispossessed,outfielderBlock,interceptionWon,wasFouled,totalContest,keyPass,shotOffTarget,totalCross,accurateCross,wonContest,blockedScoringAttempt,onTargetScoringAttempt,goodHighClaim,goalAssist,bigChanceCreated,goals,lastManTackle,errorLeadToAShot,totalOffside,bigChanceMissed,teamName,captain,match_id,match_date_str,competition,home_team_name,away_team_name,hitWoodwork,penaltyFaced,penaltyConceded,penaltyWon,punches,errorLeadToAGoal,clearanceOffLine,crossNotClaimed,penaltyMiss,penaltySave,ownGoals,penaltyShootoutSave,penaltyShootoutGoal,penaltyShootoutMiss,match_type,cohort_group,rest_days,high_congestion_flag,min_last_7d,min_last_28d,acwr_ratio,is_away,consecutive_away_games,lineup_churn
0,2023-08-20,Aaron Cresswell,NaN,NaN,aaron-cresswell,A. Cresswell,D,3.0,170.0,755,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,37,3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,11352418,2023-08-20,England Premier League,West Ham United,Chelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Domestic,Baseline,14,0,0,0,0.0,0,0,3
1,2023-08-26,Aaron Cresswell,NaN,NaN,aaron-cresswell,A. Cresswell,D,3.0,170.0,755,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,37,3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,11352466,2023-08-26,England Premier League,Brighton & Hove Albion,West Ham United,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Domestic,Baseline,6,0,0,0,0.0,1,1,2



2024-2025
./2024-2025/sofascore_dynamic/fixtureiq_dynamic_master.csv
Loaded: 7,587 rows × 117 columns
Columns:
['date', 'name', 'firstName', 'lastName', 'slug', 'shortName', 'position', 'jerseyNumber', 'height', 'userCount', 'gender', 'country', 'id', 'marketValueCurrency', 'dateOfBirthTimestamp', 'proposedMarketValueRaw', 'fieldTranslations', 'sofascoreId', 'teamId', 'shirtNumber', 'substitute', 'totalPass', 'accuratePass', 'totalLongBalls', 'accurateLongBalls', 'goalAssist', 'accurateOwnHalfPasses', 'totalOwnHalfPasses', 'accurateOppositionHalfPasses', 'totalOppositionHalfPasses', 'duelLost', 'challengeLost', 'ballRecovery', 'goodHighClaim', 'savedShotsFromInsideTheBox', 'saves', 'totalKeeperSweeper', 'accurateKeeperSweeper', 'minutesPlayed', 'touches', 'rating', 'possessionLostCtrl', 'expectedAssists', 'ratingVersions', 'totalShots', 'goalsPrevented', 'statisticsType', 'aerialLost', 'duelWon', 'shotOffTarget', 'totalClearance', 'interceptionWon', 'totalTackle', 'wonTackle', 'unsucc

,date,name,firstName,lastName,slug,shortName,position,jerseyNumber,height,userCount,gender,country,id,marketValueCurrency,dateOfBirthTimestamp,proposedMarketValueRaw,fieldTranslations,sofascoreId,teamId,shirtNumber,substitute,totalPass,accuratePass,totalLongBalls,accurateLongBalls,goalAssist,accurateOwnHalfPasses,totalOwnHalfPasses,accurateOppositionHalfPasses,totalOppositionHalfPasses,duelLost,challengeLost,ballRecovery,goodHighClaim,savedShotsFromInsideTheBox,saves,totalKeeperSweeper,accurateKeeperSweeper,minutesPlayed,touches,rating,possessionLostCtrl,expectedAssists,ratingVersions,totalShots,goalsPrevented,statisticsType,aerialLost,duelWon,shotOffTarget,totalClearance,interceptionWon,totalTackle,wonTackle,unsuccessfulTouch,wasFouled,expectedGoals,aerialWon,outfielderBlock,fouls,onTargetScoringAttempt,blockedScoringAttempt,expectedGoalsOnTarget,totalCross,accurateCross,totalOffside,dispossessed,keyPass,totalContest,wonContest,bigChanceCreated,lastManTackle,errorLeadToAShot,bigChanceMissed,goals,teamName,captain,match_id,match_date_str,competition,home_team_name,away_team_name,punches,penaltyWon,penaltyFaced,penaltyConceded,hitWoodwork,errorLeadToAGoal,ownGoals,clearanceOffLine,penaltyMiss,penaltySave,totalBallCarriesDistance,ballCarriesCount,totalProgression,progressiveBallCarriesCount,bestBallCarryProgression,totalProgressiveBallCarriesDistance,crossNotClaimed,penaltyShootoutGoal,penaltyShootoutMiss,penaltyShootoutSave,rest_days,high_congestion_flag,min_last_7d,min_last_28d,acwr_ratio,is_away,consecutive_away_games,lineup_churn,elo,home_team,home_xg,away_xg,team_xg,team_xga,xg_difference
0,2024-08-17,Aaron Cresswell,NaN,NaN,aaron-cresswell,A. Cresswell,D,3.0,170.0,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,37,3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,12436877,2024-08-17,England Premier League,West Ham United,Aston Villa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,0,0,0,0.0,0,0,0,1758.397339,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-12-29,Aaron Cresswell,NaN,NaN,aaron-cresswell,A. Cresswell,D,3.0,170.0,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,37,3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,12436513,2024-12-29,England Premier League,West Ham United,Liverpool FC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134,0,0,0,0.0,0,0,5,1758.397339,NaN,NaN,NaN,NaN,NaN,NaN



COMBINED SOFASCORE MASTER DATA
Shape: (29649, 119)

Rows by season:
2022-2023 : 7304
2023-2024 : 14758
2024-2025 : 7587

Unique matches:
732

Unique players:
643

Important workload columns:
['rest_days', 'high_congestion_flag', 'min_last_7d', 'min_last_28d', 'acwr_ratio', 'consecutive_away_games']

Important performance columns:
['rating', 'minutesPlayed', 'goals', 'goalAssist', 'totalShots', 'onTargetScoringAttempt', 'totalPass', 'accuratePass', 'keyPass', 'totalTackle', 'wonTackle', 'interceptionWon', 'duelWon', 'duelLost']


,date,name,slug,shortName,position,jerseyNumber,height,userCount,gender,country,id,marketValueCurrency,dateOfBirthTimestamp,proposedMarketValueRaw,fieldTranslations,firstName,lastName,sofascoreId,teamId,shirtNumber,substitute,totalPass,accuratePass,totalLongBalls,accurateLongBalls,accurateOwnHalfPasses,totalOwnHalfPasses,accurateOppositionHalfPasses,totalOppositionHalfPasses,aerialWon,duelWon,ballRecovery,goodHighClaim,savedShotsFromInsideTheBox,saves,minutesPlayed,touches,rating,possessionLostCtrl,expectedGoals,expectedGoalsOnTarget,expectedAssists,totalShots,goalsPrevented,statisticsType,goalAssist,totalCross,accurateCross,aerialLost,duelLost,totalClearance,interceptionWon,totalTackle,wonTackle,wasFouled,fouls,keyPass,challengeLost,totalContest,wonContest,clearanceOffLine,outfielderBlock,unsuccessfulTouch,blockedScoringAttempt,dispossessed,shotOffTarget,hitWoodwork,totalOffside,onTargetScoringAttempt,goals,penaltyWon,penaltyFaced,totalKeeperSweeper,accurateKeeperSweeper,penaltyConceded,bigChanceCreated,bigChanceMissed,teamName,captain,match_id,match_date_str,competition,home_team_name,away_team_name,lastManTackle,ownGoals,punches,crossNotClaimed,errorLeadToAGoal,errorLeadToAShot,penaltySave,penaltyMiss,ratingVersions,rest_days,high_congestion_flag,min_last_7d,min_last_28d,acwr_ratio,is_away,consecutive_away_games,lineup_churn,elo,home_team,home_xg,away_xg,team_xg,team_xga,xg_difference,penaltyShootoutSave,penaltyShootoutGoal,penaltyShootoutMiss,match_type,cohort_group,totalBallCarriesDistance,ballCarriesCount,totalProgression,progressiveBallCarriesCount,bestBallCarryProgression,totalProgressiveBallCarriesDistance
0,2022-08-07,Aaron Cresswell,aaron-cresswell,A. Cresswell,D,3.0,170.0,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,NaN,NaN,37,3,False,22.0,17.0,2.0,1.0,12.0,15.0,7.0,10.0,1.0,2.0,5.0,NaN,NaN,NaN,90,35.0,6.7,6.0,0.0,0.0,0.103592,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,3.0,2.0,NaN,1.0,4.0,2.0,1.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,10385514,2022-08-07,England Premier League,West Ham United,Manchester City,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,0,0,0,0.0,0,0,0,1751.339966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-31,Aaron Cresswell,aaron-cresswell,A. Cresswell,D,3.0,170.0,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,NaN,NaN,37,3,False,32.0,29.0,1.0,1.0,14.0,15.0,15.0,18.0,1.0,2.0,2.0,NaN,NaN,NaN,72,47.0,7.1,4.0,0.0,0.0,0.007961,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,1.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,West Ham United,NaN,10385564,2022-08-31,England Premier League,West Ham United,Tottenham Hotspur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,0,0,90,0.0,0,0,4,1751.339966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-10-19,Aaron Cresswell,aaron-cresswell,A. Cresswell,D,3.0,170.0,751,M,"{'alpha2': 'EN', 'alpha3': 'ENG', 'name': 'England', 'slug': 'england'}",43443,EUR,629683200,"{'value': 480000, 'currency': 'EUR'}","{'nameTranslation': {'ar': 'آرون كريسويل', 'bn': 'অ্যারন ক্রেসওয়েল', 'hi': ...",NaN,NaN,NaN,37,3,False,67.0,54.0,10.0,6.0,29.0,35.0,27.0,40.0,NaN,4.0,7.0,NaN,NaN,NaN,90,98.0,7.8,22.0,0.0,0.0,0.181447,0,NaN,"{'sportSlug': 'football', 'statisticsType': 'player'}",NaN,8.0,2.0,NaN,2.0,1.0,3.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,West Ham United,NaN,10385729,2022-10-19,England Premier League,Liverpool FC,West Ham United,NaN,

## Section 5 — Data Source D: FBRef Detailed Match Stats
Per-match **action stats** (goals, shots, tackles, crosses, fouls, cards…) scraped from FBRef.
**Coverage is limited to 4 CL teams per season** (teams were selected because they played Champions League).


In [14]:
# Section 5 — FBRef Detailed Match Stats
# Per-match action stats for selected Champions League teams
# Coverage is limited to 4 teams per season
# Nothing is saved

import os
import pandas as pd

BASE_DIR = "."

fbref_teams = {
    "2022-2023": [
        "chelsea_2022_2023",
        "liverpool_2022_2023",
        "manchester_city_2022_2023",
        "tottenham_hotspur_2022_2023",
    ],
    "2023-2024": [
        "arsenal_2023_2024",
        "manchester_city_2023_2024",
        "manchester_united_2023_2024",
        "newcastle_united_2023_2024",
    ],
    "2024-2025": [
        "arsenal_2024_2025",
        "aston_villa_2024_2025",
        "liverpool_2024_2025",
        "manchester_city_2024_2025",
    ],
}

fbref_dfs = {}

for season, teams in fbref_teams.items():
    season_rows = []

    print("\n" + "=" * 80)
    print(season)

    for team in teams:
        path = os.path.join(
            BASE_DIR,
            season,
            "fbref",
            team,
            "match_reports",
            "master_player_stats.csv"
        )

        print(path)

        if os.path.exists(path):
            df = pd.read_csv(path, parse_dates=["match_date"])
            df["source_team_folder"] = team
            df["source_season"] = season
            season_rows.append(df)

            print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
        else:
            print("File not found")

    if season_rows:
        fbref_dfs[season] = pd.concat(season_rows, ignore_index=True)

# Combine all seasons
fbref_all = pd.concat(fbref_dfs.values(), ignore_index=True)

print("\n" + "=" * 80)
print("COMBINED FBREF DATA")
print("=" * 80)

print("Shape:", fbref_all.shape)

print("\nRows by season:")
print(fbref_all["source_season"].value_counts().sort_index())

print("\nTeams:")
print(fbref_all["source_team_folder"].value_counts())

print("\nColumns:")
print(list(fbref_all.columns))

print("\nCompetitions:")
print(fbref_all["competition"].value_counts())

print("\nSample:")
display(fbref_all.head(3))


2022-2023
./2022-2023/fbref/chelsea_2022_2023/match_reports/master_player_stats.csv
Loaded: 778 rows × 41 columns
./2022-2023/fbref/liverpool_2022_2023/match_reports/master_player_stats.csv
Loaded: 805 rows × 41 columns
./2022-2023/fbref/manchester_city_2022_2023/match_reports/master_player_stats.csv
Loaded: 872 rows × 41 columns
./2022-2023/fbref/tottenham_hotspur_2022_2023/match_reports/master_player_stats.csv
Loaded: 732 rows × 41 columns

2023-2024
./2023-2024/fbref/arsenal_2023_2024/match_reports/master_player_stats.csv
Loaded: 769 rows × 41 columns
./2023-2024/fbref/manchester_city_2023_2024/match_reports/master_player_stats.csv
Loaded: 805 rows × 41 columns
./2023-2024/fbref/manchester_united_2023_2024/match_reports/master_player_stats.csv
Loaded: 783 rows × 41 columns
./2023-2024/fbref/newcastle_united_2023_2024/match_reports/master_player_stats.csv
Loaded: 759 rows × 41 columns

2024-2025
./2024-2025/fbref/arsenal_2024_2025/match_reports/master_player_stats.csv
Loaded: 866 ro

,season,tracked_team,match_date,competition,opponent_team,original_file,Match_ID,Match_Report_URL,Date,Competition,Round,Venue,Opponent,Result,Match_Report_Name,Team,Player,shirtnumber,Nation,Pos,Age,Min,Gls,Ast,PK,PKatt,shots,shots_on_target,CrdY,CrdR,fouls,fouled,offsides,crosses,tackles_won,interceptions,own_goals,pens_won,pens_conceded,source_team_folder,source_season
0,2022-2023,Chelsea 2022 2023,2022-08-06,Premier League,Everton,2022-08-06_premier_league_everton_player_stats.csv,3a917cee,https://fbref.com/en/matches/3a917cee/Everton-Chelsea-August-6-2022-Premier-...,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,2022-08-06_premier_league_everton,Chelsea,Raheem Sterling,17,eng ENG,FW,27-241,90,0,0,0,0,3.0,0.0,0,0,3.0,3.0,1.0,0.0,1.0,0.0,0.0,NaN,NaN,chelsea_2022_2023,2022-2023
1,2022-2023,Chelsea 2022 2023,2022-08-06,Premier League,Everton,2022-08-06_premier_league_everton_player_stats.csv,3a917cee,https://fbref.com/en/matches/3a917cee/Everton-Chelsea-August-6-2022-Premier-...,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,2022-08-06_premier_league_everton,Chelsea,Kai Havertz,29,de GER,FW,23-056,74,0,0,0,0,3.0,1.0,0,0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,chelsea_2022_2023,2022-2023
2,2022-2023,Chelsea 2022 2023,2022-08-06,Premier League,Everton,2022-08-06_premier_league_everton_player_stats.csv,3a917cee,https://fbref.com/en/matches/3a917cee/Everton-Chelsea-August-6-2022-Premier-...,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,2022-08-06_premier_league_everton,Chelsea,Armando Broja,18,al ALB,FW,20-330,16,0,0,0,0,1.0,1.0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,chelsea_2022_2023,2022-2023


In [ ]:
# Section 5D2 — FBRef Season Aggregates + Team Match List
# Example: Chelsea 2022-2023
# Nothing is saved

BASE_DIR = "."

sample_team_dir = os.path.join(
    BASE_DIR,
    "2022-2023",
    "fbref",
    "chelsea_2022_2023"
)

path_all_comps = os.path.join(
    sample_team_dir,
    "chelsea_2022_2023_players_all_competitions.csv"
)

path_matches = os.path.join(
    sample_team_dir,
    "chelsea_2022_2023_matches_all.csv"
)

print("=" * 80)
print("FBREF SEASON AGGREGATE PER PLAYER")
print("=" * 80)
print(path_all_comps)

if os.path.exists(path_all_comps):
    df_all_comps = pd.read_csv(path_all_comps)

    print("Shape:", df_all_comps.shape)
    print("Columns:")
    print(list(df_all_comps.columns))

    print("\nSample:")
    display(df_all_comps.head(3))
else:
    print("File not found")


print("\n" + "=" * 80)
print("FBREF TEAM MATCH LIST")
print("=" * 80)
print(path_matches)

if os.path.exists(path_matches):
    df_matches = pd.read_csv(path_matches, parse_dates=["Date"])

    print("Shape:", df_matches.shape)
    print("Columns:")
    print(list(df_matches.columns))

    print("\nCompetitions:")
    print(df_matches["Competition"].value_counts())

    print("\nSample:")
    display(df_matches.head(3))
else:
    print("File not found")


FBREF SEASON AGGREGATE PER PLAYER
./2022-2023/fbref/chelsea_2022_2023/chelsea_2022_2023_players_all_competitions.csv
Shape: (41, 22)
Columns:
['Player', 'Nation', 'Pos', 'Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls_90', 'Ast_90', 'G+A_90', 'G-PK_90', 'G+A-PK_90', 'Matches']

Sample:


,Player,Nation,Pos,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Gls_90,Ast_90,G+A_90,G-PK_90,G+A-PK_90,Matches
0,Kepa Arrizabalaga,esESP,GK,27,39,39,"3,465",38.5,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.00,0.00,0.00,0.00,0.00,Matches
1,Kai Havertz,deGER,"FW,MF",23,47,38,"3,249",36.1,9.0,1.0,10.0,7.0,2.0,3.0,5.0,0.0,0.25,0.03,0.28,0.19,0.22,Matches
2,Thiago Silva,brBRA,DF,37,35,33,"2,992",33.2,0.0,2.0,2.0,0.0,0.0,0.0,4.0,0.0,0.00,0.06,0.06,0.00,0.06,Matches



FBREF TEAM MATCH LIST
./2022-2023/fbref/chelsea_2022_2023/chelsea_2022_2023_matches_all.csv
Shape: (50, 19)
Columns:
['Date', 'start_time', 'Competition', 'Round', 'Day', 'Venue', 'Result', 'goals_for', 'goals_against', 'Opponent', 'possession', 'attendance', 'captain', 'formation', 'opp_formation', 'referee', 'Match_Report_URL', 'Match_Report', 'notes']

Competitions:
Competition
Premier League    38
Champions Lg      10
EFL Cup            1
FA Cup             1
Name: count, dtype: int64

Sample:


,Date,start_time,Competition,Round,Day,Venue,Result,goals_for,goals_against,Opponent,possession,attendance,captain,formation,opp_formation,referee,Match_Report_URL,Match_Report,notes
0,2022-08-06,17:30(18:30),Premier League,Matchweek 1,Sat,Away,W,1,0,Everton,63,"39,254",César Azpilicueta,3-4-3,5-4-1,Craig Pawson,https://fbref.com/en/matches/3a917cee/Everton-Chelsea-August-6-2022-Premier-...,Match Report,NaN
1,2022-08-14,16:30(17:30),Premier League,Matchweek 2,Sun,Home,D,2,2,Tottenham Hotspur,64,"39,946",Jorginho,3-4-3,3-4-3,Anthony Taylor,https://fbref.com/en/matches/01e57bf5/Chelsea-Tottenham-Hotspur-August-14-20...,Match Report,NaN
2,2022-08-21,14:00(15:00),Premier League,Matchweek 3,Sun,Away,L,0,3,Leeds United,60,"36,372",Jorginho,3-5-2,4-2-3-1,Stuart Attwell,https://fbref.com/en/matches/a5632124/Leeds-United-Chelsea-August-21-2022-Pr...,Match Report,NaN


## Section 6 — Data Source E: Injury Data (5 files)
Five complementary injury files at different granularities:
- **ALL_TEAMS injuries_days_out**: Row per player per missed fixture (most granular)
- **estimated_injury_spells**: Row per injury episode with type & dates
- **player_season_absence_burden**: Season totals per player
- **team_match_injury_outcomes**: Per-match team injury burden (squad missing players count)
- **team_season_injury_burden**: Season totals per team


In [16]:
# Section 6 — Injury Data
# Five complementary injury files at different granularities
# Nothing is saved

import os
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

injury_data = {
    "ALL_TEAMS_injuries_days_out": {},
    "estimated_injury_spells": {},
    "player_season_absence_burden": {},
    "team_match_injury_outcomes": {},
    "team_season_injury_burden": {},
}

for season in SEASONS:
    season_key = season.replace("-", "_")

    raw_injury_paths = [
        os.path.join(
            BASE_DIR,
            f"SEASON_{season_key}",
            "injuries",
            f"ALL_TEAMS_{season}_injuries_days_out.csv"
        ),
        os.path.join(
            BASE_DIR,
            season,
            "injuries",
            f"ALL_TEAMS_{season}_injuries_days_out.csv"
        ),
    ]

    processed_injury_dir = os.path.join(
        BASE_DIR,
        f"SEASON_{season_key}",
        "processed_injuries"
    )

    processed_paths = {
        "estimated_injury_spells": os.path.join(
            processed_injury_dir,
            f"estimated_injury_spells_{season}.csv"
        ),
        "player_season_absence_burden": os.path.join(
            processed_injury_dir,
            f"player_season_absence_burden_{season}.csv"
        ),
        "team_match_injury_outcomes": os.path.join(
            processed_injury_dir,
            f"team_match_injury_outcomes_{season}.csv"
        ),
        "team_season_injury_burden": os.path.join(
            processed_injury_dir,
            f"team_season_injury_burden_{season}.csv"
        ),
    }

    print("\n" + "=" * 80)
    print(season)

    # Load ALL_TEAMS file
    all_teams_loaded = False

    for path in raw_injury_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            injury_data["ALL_TEAMS_injuries_days_out"][season] = df
            all_teams_loaded = True

            print("\nALL_TEAMS_injuries_days_out")
            print(path)
            print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
            print("Columns:", list(df.columns))
            display(df.head(2))
            break

    if not all_teams_loaded:
        print("\nALL_TEAMS_injuries_days_out")
        print("File not found")

    # Load processed files
    for key, path in processed_paths.items():
        print("\n" + "-" * 80)
        print(key)
        print(path)

        if os.path.exists(path):
            df = pd.read_csv(path)
            injury_data[key][season] = df

            print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
            print("Columns:", list(df.columns))
            display(df.head(2))
        else:
            print("File not found")


# Combine each injury table across seasons
injury_all = {}

print("\n" + "=" * 80)
print("COMBINED INJURY TABLES")
print("=" * 80)

for key, season_dict in injury_data.items():
    if season_dict:
        combined = pd.concat(
            [
                df.assign(source_season=season)
                for season, df in season_dict.items()
            ],
            ignore_index=True
        )

        injury_all[key] = combined

        print("\n" + key)
        print("Shape:", combined.shape)
        print("Rows by season:")
        print(combined["source_season"].value_counts().sort_index())

print("\nCreated in memory:")
print("injury_data")
print("injury_all")

print("\nNothing was saved.")


2022-2023

ALL_TEAMS_injuries_days_out
./SEASON_2022_2023/injuries/ALL_TEAMS_2022-2023_injuries_days_out.csv
Loaded: 2,773 rows × 9 columns
Columns: ['season', 'team_name', 'player_id', 'player_name', 'injury_type', 'reason', 'fixture_date', 'end_date', 'days_out']


,season,team_name,player_id,player_name,injury_type,reason,fixture_date,end_date,days_out
0,2022-2023,Arsenal,727,R. Nelson,Missing Fixture,Injury,2022-08-13,NaN,182
1,2022-2023,Arsenal,3246,N. Pepe,Missing Fixture,Inactive,2022-08-20,NaN,0



--------------------------------------------------------------------------------
estimated_injury_spells
./SEASON_2022_2023/processed_injuries/estimated_injury_spells_2022-2023.csv
Loaded: 380 rows × 11 columns
Columns: ['Season', 'Season_Label', 'Team', 'Player', 'Player_ID', 'Reason', 'Injury_Category', 'First_Missed_Fixture', 'Last_Missed_Fixture', 'Matches_Missed', 'Reported_Days_Out']


,Season,Season_Label,Team,Player,Player_ID,Reason,Injury_Category,First_Missed_Fixture,Last_Missed_Fixture,Matches_Missed,Reported_Days_Out
0,2022-2023,2022/23,Arsenal,R. Nelson,727,Injury,Generic injury,2022-08-13,2023-08-12,14,182
1,2022-2023,2022/23,Arsenal,N. Pepe,3246,Inactive,Other,2022-08-20,2022-08-20,1,0



--------------------------------------------------------------------------------
player_season_absence_burden
./SEASON_2022_2023/processed_injuries/player_season_absence_burden_2022-2023.csv
Loaded: 351 rows × 12 columns
Columns: ['Season', 'Season_Label', 'Team', 'Player', 'Player_ID', 'Matches_Missed_Total', 'Matches_Missed_Injury', 'First_Missed_Fixture', 'Last_Missed_Fixture', 'Main_Reason', 'Main_Injury_Category', 'Max_Reported_Days_Out']


...


--------------------------------------------------------------------------------
team_match_injury_outcomes
./SEASON_2024_2025/processed_injuries/team_match_injury_outcomes_2024-2025.csv
Loaded: 760 rows × 10 columns
Columns: ['Season', 'Season_Label', 'Team', 'Fixture_Date', 'Players_Missing_Total', 'Players_Missing_Injury', 'Players_Missing_Non_Injury', 'Soft_Tissue_Missing', 'Max_Reported_Days_Out', 'Mean_Reported_Days_Out']


,Season,Season_Label,Team,Fixture_Date,Players_Missing_Total,Players_Missing_Injury,Players_Missing_Non_Injury,Soft_Tissue_Missing,Max_Reported_Days_Out,Mean_Reported_Days_Out
0,2024-2025,2024/25,Manchester United,2024-08-16,5,5,0,1,281,124.4
1,2024-2025,2024/25,Fulham,2024-08-16,0,0,0,0,0,0.0



--------------------------------------------------------------------------------
team_season_injury_burden
./SEASON_2024_2025/processed_injuries/team_season_injury_burden_2024-2025.csv
Loaded: 20 rows × 11 columns
Columns: ['Season', 'Season_Label', 'Team', 'Total_Player_Fixture_Absences', 'Total_Injury_Player_Fixture_Absences', 'Mean_Injury_Players_Missing_Per_Match', 'Total_Soft_Tissue_Missing', 'Mean_Soft_Tissue_Missing_Per_Match', 'Unique_Players_Missing', 'Max_Injury_Players_Missing_In_Match', 'Total_Matches']


,Season,Season_Label,Team,Total_Player_Fixture_Absences,Total_Injury_Player_Fixture_Absences,Mean_Injury_Players_Missing_Per_Match,Total_Soft_Tissue_Missing,Mean_Soft_Tissue_Missing_Per_Match,Unique_Players_Missing,Max_Injury_Players_Missing_In_Match,Total_Matches
0,2024-2025,2024/25,Arsenal,221,210,5.53,76,2.00,21,6,38
1,2024-2025,2024/25,Aston Villa,119,111,2.92,52,1.37,24,5,38



COMBINED INJURY TABLES

ALL_TEAMS_injuries_days_out
Shape: (9526, 10)
Rows by season:
source_season
2022-2023    2773
2023-2024    3600
2024-2025    3153
Name: count, dtype: int64

estimated_injury_spells
Shape: (1256, 12)
Rows by season:
source_season
2022-2023    380
2023-2024    475
2024-2025    401
Name: count, dtype: int64

player_season_absence_burden
Shape: (1160, 13)
Rows by season:
source_season
2022-2023    351
2023-2024    408
2024-2025    401
Name: count, dtype: int64

team_match_injury_outcomes
Shape: (2280, 11)
Rows by season:
source_season
2022-2023    760
2023-2024    760
2024-2025    760
Name: count, dtype: int64

team_season_injury_burden
Shape: (60, 12)
Rows by season:
source_season
2022-2023    20
2023-2024    20
2024-2025    20
Name: count, dtype: int64

Created in memory:
injury_data
injury_all

Nothing was saved.


## Section 7 — Complete Data Inventory Summary
A consolidated table of every dataset: rows, columns, what it provides, and the merge keys available.


In [ ]:
# Section 7 — Complete Data Inventory Summary
# Consolidated summary of all loaded sources


rows = []

# A — API Player Stats
for season, df in api_player_dfs.items():
    rows.append({
        "Season": season,
        "Source": "API Player Stats",
        "Granularity": "per-match, per-player",
        "Coverage": "All 20 PL teams + all competitions",
        "Rows": df.shape[0],
        "Cols": df.shape[1],
        "Merge Keys": "fixture_id, date, player_team, player_id/player_name"
    })

# B — API Team Results
for season, df in api_team_dfs.items():
    rows.append({
        "Season": season,
        "Source": "API Team Results",
        "Granularity": "per-match, per-team",
        "Coverage": "All 20 PL teams + all competitions",
        "Rows": df.shape[0],
        "Cols": df.shape[1],
        "Merge Keys": "fixture_id, date, team_name"
    })

# C — SofaScore Dynamic Clean
for season, df in sofascore_dynamic.items():
    rows.append({
        "Season": season,
        "Source": "SofaScore Dynamic Clean",
        "Granularity": "per-match, per-player",
        "Coverage": "All 20 PL teams, Premier League only",
        "Rows": df.shape[0],
        "Cols": df.shape[1],
        "Merge Keys": "match_date_str, teamName, name"
    })

# C2 — SofaScore Master
for season, df in sofascore_master.items():
    rows.append({
        "Season": season,
        "Source": "SofaScore Master",
        "Granularity": "per-match, per-player",
        "Coverage": "All 20 PL teams, Premier League only",
        "Rows": df.shape[0],
        "Cols": df.shape[1],
        "Merge Keys": "match_date_str, teamName, name"
    })

# D — FBRef Match Stats
for season, df in fbref_dfs.items():
    rows.append({
        "Season": season,
        "Source": "FBRef Match Stats",
        "Granularity": "per-match, per-player",
        "Coverage": "4 selected CL teams per season",
        "Rows": df.shape[0],
        "Cols": df.shape[1],
        "Merge Keys": "match_date, Team, Player"
    })

# E — Injury files
injury_labels = {
    "ALL_TEAMS_injuries_days_out": {
        "Granularity": "per missed fixture, per player",
        "Coverage": "All 20 teams",
        "Merge Keys": "season, fixture_date, team_name, player_name/player_id"
    },
    "estimated_injury_spells": {
        "Granularity": "per injury spell",
        "Coverage": "All 20 teams",
        "Merge Keys": "Season, Team, Player/Player_ID, date range"
    },
    "player_season_absence_burden": {
        "Granularity": "per season, per player",
        "Coverage": "All 20 teams",
        "Merge Keys": "Season, Team, Player/Player_ID"
    },
    "team_match_injury_outcomes": {
        "Granularity": "per match, per team",
        "Coverage": "All 20 teams, Premier League only",
        "Merge Keys": "Season, Fixture_Date, Team"
    },
    "team_season_injury_burden": {
        "Granularity": "per season, per team",
        "Coverage": "All 20 teams",
        "Merge Keys": "Season, Team"
    },
}

for file_key, seasons_dict in injury_data.items():
    for season, df in seasons_dict.items():
        rows.append({
            "Season": season,
            "Source": f"Injury: {file_key}",
            "Granularity": injury_labels[file_key]["Granularity"],
            "Coverage": injury_labels[file_key]["Coverage"],
            "Rows": df.shape[0],
            "Cols": df.shape[1],
            "Merge Keys": injury_labels[file_key]["Merge Keys"]
        })

summary_df = pd.DataFrame(rows)

summary_df = summary_df.sort_values(["Season", "Source"]).reset_index(drop=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

print("=" * 100)
print("COMPLETE DATA INVENTORY SUMMARY")
print("=" * 100)

display(summary_df)

print("\nRows by source:")
display(
    summary_df
    .groupby("Source")[["Rows"]]
    .sum()
    .sort_values("Rows", ascending=False)
)

print("\nColumns by source:")
display(
    summary_df
    .groupby("Source")[["Cols"]]
    .max()
    .sort_values("Cols", ascending=False)
)

print("\nNothing was saved.")

COMPLETE DATA INVENTORY SUMMARY


,Season,Source,Granularity,Coverage,Rows,Cols,Merge Keys
0,2022-2023,API Player Stats,"per-match, per-player",All 20 PL teams + all competitions,23455,34,"fixture_id, date, player_team, player_id/player_name"
1,2022-2023,API Team Results,"per-match, per-team",All 20 PL teams + all competitions,2802,49,"fixture_id, date, team_name"
2,2022-2023,FBRef Match Stats,"per-match, per-player",4 selected CL teams per season,3187,41,"match_date, Team, Player"
3,2022-2023,Injury: ALL_TEAMS_injuries_days_out,"per missed fixture, per player",All 20 teams,2773,9,"season, fixture_date, team_name, player_name/player_id"
4,2022-2023,Injury: estimated_injury_spells,per injury spell,All 20 teams,380,11,"Season, Team, Player/Player_ID, date range"
5,2022-2023,Injury: player_season_absence_burden,"per season, per player",All 20 teams,351,12,"Season, Team, Player/Player_ID"
6,2022-2023,Injury: team_match_injury_outcomes,"per match, per team","All 20 teams, Premier League only",760,10,"Season, Fixture_Date, Team"
7,2022-2023,Injury: team_season_injury_burden,"per season, per team",All 20 teams,20,11,"Season, Team"
8,2022-2023,SofaScore Dynamic Clean,"per-match, per-player","All 20 PL teams, Premier League only",6500,19,"match_date_str, teamName, name"
9,2022-2023,SofaScore Master,"per-match, per-player","All 20 PL teams, Premier League only",7304,108,"match_date_str, teamName, name"



Rows by source:


,Rows
Source,
API Player Stats,68722
SofaScore Master,29649
SofaScore Dynamic Clean,26368
FBRef Match Stats,9739
Injury: ALL_TEAMS_injuries_days_out,9526
API Team Results,5066
Injury: team_match_injury_outcomes,2280
Injury: estimated_injury_spells,1256
Injury: player_season_absence_burden,1160



Columns by source:


,Cols
Source,
SofaScore Master,117
API Team Results,53
FBRef Match Stats,41
API Player Stats,34
SofaScore Dynamic Clean,19
Injury: player_season_absence_burden,12
Injury: team_season_injury_burden,11
Injury: estimated_injury_spells,11
Injury: team_match_injury_outcomes,10



Nothing was saved.


## Section 8 — Integration Plan: Building the Master Training Table

**Objective**: One row = one player × one match, enriched with all available context.

### Primary Merge: API Player Stats → base table
The `multi_competition_player_stats` file is the base because it covers:
- All 3 seasons
- All 20 PL teams
- All competitions (PL + CL + FA/LC)

### Enrichments via LEFT JOIN:

| JOIN | Key | What it adds |
|------|-----|-------------|
| **API Team Results** | `fixture_id` | Match-level team stats (xG, possession, shots) |
| **SofaScore Dynamic** | `(date, player_name, team)` | `rest_days`, `acwr_ratio`, `min_last_7d`, `high_congestion_flag` |
| **FBRef Match Stats** | `(match_date, player, team)` | `shots`, `tackles_won`, `crosses`, `fouls`, `interceptions` |
| **Injury: team_match_outcomes** | `(date, team)` | Squad injury burden on that matchday |
| **Injury: ALL_TEAMS_days_out** | `(fixture_date, player_name, team)` | Was *this specific player* injured around this date? |

### Target Variable Options:
- **Regression**: `rating` of next match (performance level)
- **Classification**: Binary flag — rating drops ≥ 1 point in next match
- **Multi-horizon**: Predict decline over next 3 matches (rolling window)



In [18]:
# Section 8 — Integration Plan: Build Master Training Table
# One row = one player × one match


def clean_name(x):
    if pd.isna(x):
        return np.nan
    return (
        str(x)
        .lower()
        .strip()
        .replace(".", "")
        .replace("-", " ")
        .replace("_", " ")
        .replace("&", "and")
    )

# ============================================================
# 1. BASE TABLE: API PLAYER STATS
# ============================================================

master = api_players_all.copy()

master["date"] = pd.to_datetime(master["date"], errors="coerce")
master["team_key"] = master["player_team"].map(clean_name)
master["player_key"] = master["player_name"].map(clean_name)

print("=" * 80)
print("BASE TABLE: API PLAYER STATS")
print("=" * 80)

print("Shape:", master.shape)
print("Unique fixtures:", master["fixture_id"].nunique())
print("Unique players:", master["player_id"].nunique())
print("Unique teams:", master["player_team"].nunique())

print("\nRows by season:")
print(master["season"].value_counts().sort_index())

print("\nRows by competition:")
print(master["competition"].value_counts())

display(master.head())



BASE TABLE: API PLAYER STATS
Shape: (68722, 36)
Unique fixtures: 1715
Unique players: 5289
Unique teams: 152

Rows by season:
season
2022    23455
2023    22021
2024    23246
Name: count, dtype: int64

Rows by competition:
competition
Premier League      45552
League Cup          10997
FA Cup               6972
Champions League     5081
Community Shield      120
Name: count, dtype: int64


,fixture_id,date,competition,season,round,home_team,away_team,player_team,player_id,player_name,player_number,player_position,minutes_played,rating,is_captain,is_substitute,shots_total,shots_on_target,goals,assists,passes_total,passes_key,passes_accuracy,dribbles_attempts,dribbles_success,tackles_total,tackles_blocks,tackles_interceptions,duels_total,duels_won,fouls_drawn,fouls_committed,cards_yellow,cards_red,team_key,player_key
0,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18835,Vicente Guaita,13,G,90,6.2,False,False,0,0,0,0,37,0,35.0,0,0,0,0,0,0,0,0,0,0,0,crystal palace,vicente guaita
1,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,18862,Nathaniel Clyne,17,D,90,6.3,False,False,0,0,0,0,56,1,50.0,1,1,0,0,1,5,3,1,1,1,0,crystal palace,nathaniel clyne
2,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,2729,Joachim Andersen,16,D,90,7.9,False,False,0,0,0,0,104,1,91.0,0,0,6,0,2,15,12,2,1,0,0,crystal palace,joachim andersen
3,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,67971,Marc Guéhi,6,D,90,6.3,True,False,0,0,0,0,105,0,93.0,0,0,0,2,1,7,1,0,1,0,0,crystal palace,marc guéhi
4,867946,2022-08-05 19:00:00+00:00,Premier League,2022,Regular Season - 1,Crystal Palace,Arsenal,Crystal Palace,182201,Tyrick Mitchell,3,D,90,6.3,False,False,0,0,0,0,52,0,39.0,1,0,1,1,2,7,2,1,2,0,0,crystal palace,tyrick mitchell


## Section 9 — Build Master Training Table → `Fixture_IQ_Data_Seasons.csv`

Executes all 7 build steps end-to-end:

| Step | Action | Join Key |
|------|--------|----------|
| 1 | Stack API Player Stats (base, 3 seasons) | — |
| 2 | LEFT JOIN API Team Results | `fixture_id + team` |
| 3 | LEFT JOIN SofaScore Fatigue Metrics | `date + player + team` |
| 4 | LEFT JOIN FBRef Action Stats | `date + player + team` |
| 5 | LEFT JOIN Squad Injury Burden | `date + team` |
| 6 | LEFT JOIN Individual Injury Flag | `date + player + team` |
| 7 | Engineer target variable | `next_sofascore_rating` per player |


In [24]:
# ============================================================
# SECTION 9 — Build Master Training Table
# Output: Fixture_IQ_Data_Seasons_2022-2025.csv
# One row = one player × one match
# All 3 seasons, all competitions
#
# Integrated:
#   - API player base
#   - API team results
#   - SofaScore ratings / sparse advanced info
#   - FBRef selected-team action stats
#   - Squad injury burden
#   - Workload/fatigue features for ALL rows
#   - Lag-based individual injury features
#   - Player-season injury burden
#   - Target variables
# ============================================================

import os
import unicodedata
from bisect import bisect_left

import numpy as np
import pandas as pd

BASE_DIR = "."
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

OUT_PATH = os.path.join(BASE_DIR, "Fixture_IQ_Data_Seasons_2022-2025.csv")


# ============================================================
# Utilities
# ============================================================

def norm(s):
    """Lowercase + strip accents + remove punctuation."""
    if pd.isna(s):
        return ""

    s = unicodedata.normalize("NFD", str(s).lower().strip())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")

    return (
        s.replace("'", "")
         .replace("-", " ")
         .replace(".", "")
         .replace("_", " ")
         .replace("&", "and")
         .strip()
    )


def to_date(series):
    """Parse dates to timezone-naive midnight-normalised date."""
    dt = pd.to_datetime(series, errors="coerce", utc=True)
    return dt.dt.tz_convert(None).dt.normalize()


def pct_float(series):
    """Convert '56%' strings to 56.0 floats."""
    return pd.to_numeric(series.astype(str).str.rstrip("%"), errors="coerce")


# ============================================================
# STEP 1 — Base: API Player Stats
# ============================================================

bases = []

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"API_SEASON_{season_key}",
        f"multi_competition_player_stats_{season_key}.csv"
    )

    df = pd.read_csv(path)
    bases.append(df)

master = pd.concat(bases, ignore_index=True)

master["date"] = to_date(master["date"])
master["_np"] = master["player_name"].map(norm)
master["_nt"] = master["player_team"].map(norm)

master["season_key"] = master["season"].astype(str).replace({
    "2022": "2022-2023",
    "2023": "2023-2024",
    "2024": "2024-2025",
    "2022-2023": "2022-2023",
    "2023-2024": "2023-2024",
    "2024-2025": "2024-2025",
})

print("=" * 80)
print("STEP 1 — BASE: API PLAYER STATS")
print("=" * 80)
print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
print("\nRows by competition:")
print(master["competition"].value_counts())


# ============================================================
# STEP 2 — API Team Results
# Join key: fixture_id + team
# ============================================================

team_results = []

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"API_SEASON_{season_key}",
        f"multi_competition_team_results_{season_key}.csv"
    )

    df = pd.read_csv(path)
    team_results.append(df)

tr = pd.concat(team_results, ignore_index=True)

for col in [
    "home_ball_possession",
    "away_ball_possession",
    "home_passes_%",
    "away_passes_%",
]:
    if col in tr.columns:
        tr[col] = pct_float(tr[col])

for stat, home_col, away_col in [
    ("shots_on_goal", "home_shots_on_goal", "away_shots_on_goal"),
    ("total_shots", "home_total_shots", "away_total_shots"),
    ("possession", "home_ball_possession", "away_ball_possession"),
    ("corner_kicks", "home_corner_kicks", "away_corner_kicks"),
    ("fouls", "home_fouls", "away_fouls"),
    ("gk_saves", "home_goalkeeper_saves", "away_goalkeeper_saves"),
    ("expected_goals", "home_expected_goals", "away_expected_goals"),
]:
    if home_col in tr.columns and away_col in tr.columns:
        tr[f"team_{stat}"] = np.where(tr["is_home"], tr[home_col], tr[away_col])
        tr[f"opp_{stat}"] = np.where(tr["is_home"], tr[away_col], tr[home_col])

keep = [
    "fixture_id",
    "team_name",
    "is_home",
    "opponent",
    "goals_for",
    "goals_against",
    "result",
    "points",
    "team_shots_on_goal",
    "team_total_shots",
    "team_possession",
    "team_corner_kicks",
    "team_fouls",
    "team_gk_saves",
    "team_expected_goals",
    "opp_shots_on_goal",
    "opp_total_shots",
    "opp_possession",
    "opp_expected_goals",
]

keep = [c for c in keep if c in tr.columns]

tr_slim = tr[keep].copy()
tr_slim["_nt"] = tr_slim["team_name"].map(norm)

tr_slim = (
    tr_slim
    .drop(columns=["team_name"])
    .rename(columns={"opponent": "opponent_team"})
    .drop_duplicates(subset=["fixture_id", "_nt"])
)

before = len(master)

master = master.merge(
    tr_slim,
    on=["fixture_id", "_nt"],
    how="left",
    validate="many_to_one"
)

print("\n" + "=" * 80)
print("STEP 2 — API TEAM RESULTS")
print("=" * 80)
print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
print("Row count preserved:", len(master) == before)
print(f"Matched team rows: {master['goals_for'].notna().sum():,}/{len(master):,}")


# ============================================================
# STEP 3 — SofaScore Dynamic Master
# Join key: date + player + team
# Premier League only
#
# We keep SofaScore rating and source-specific minutes where available.
# Workload features are recomputed later for ALL rows.
# ============================================================

ss_cols = [
    "match_date_str",
    "name",
    "teamName",
    "rest_days",
    "acwr_ratio",
    "min_last_7d",
    "min_last_28d",
    "high_congestion_flag",
    "consecutive_away_games",
    "minutesPlayed",
    "rating",
]

ss_list = []

for season in SEASONS:
    path = os.path.join(
        BASE_DIR,
        season,
        "sofascore_dynamic",
        "fixtureiq_dynamic_master.csv"
    )

    if os.path.exists(path):
        df = pd.read_csv(path, usecols=lambda c: c in ss_cols)
        ss_list.append(df)

if ss_list:
    ss = pd.concat(ss_list, ignore_index=True)

    ss["match_date_str"] = to_date(ss["match_date_str"])
    ss["_np"] = ss["name"].map(norm)
    ss["_nt"] = ss["teamName"].map(norm)

    if "rest_days" in ss.columns:
        ss["rest_days"] = ss["rest_days"].clip(upper=21)

    ss = ss.rename(columns={
        "rating": "sofascore_rating",
        "minutesPlayed": "ss_minutes",
        "rest_days": "sofascore_rest_days",
        "acwr_ratio": "sofascore_acwr_ratio",
        "min_last_7d": "sofascore_min_last_7d",
        "min_last_28d": "sofascore_min_last_28d",
        "high_congestion_flag": "sofascore_high_congestion_flag",
        "consecutive_away_games": "sofascore_consecutive_away_games",
    })

    ss = ss.drop(columns=["name", "teamName"])
    ss = ss.drop_duplicates(subset=["match_date_str", "_np", "_nt"])

    before = len(master)

    master = master.merge(
        ss,
        left_on=["date", "_np", "_nt"],
        right_on=["match_date_str", "_np", "_nt"],
        how="left",
        validate="many_to_one"
    )

    master = master.drop(columns=["match_date_str"])

    print("\n" + "=" * 80)
    print("STEP 3 — SOFASCORE DYNAMIC MASTER")
    print("=" * 80)
    print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
    print("Row count preserved:", len(master) == before)
    print(f"Matched SofaScore rows: {master['sofascore_rating'].notna().sum():,}/{len(master):,}")

else:
    print("\nSTEP 3 — No SofaScore files found")


# ============================================================
# STEP 4 — FBRef Match Stats
# Join key: date + player + team
# Selected teams only
# ============================================================

fbref_teams = {
    "2022-2023": [
        "chelsea_2022_2023",
        "liverpool_2022_2023",
        "manchester_city_2022_2023",
        "tottenham_hotspur_2022_2023",
    ],
    "2023-2024": [
        "arsenal_2023_2024",
        "manchester_city_2023_2024",
        "manchester_united_2023_2024",
        "newcastle_united_2023_2024",
    ],
    "2024-2025": [
        "arsenal_2024_2025",
        "aston_villa_2024_2025",
        "liverpool_2024_2025",
        "manchester_city_2024_2025",
    ],
}

fb_cols = [
    "match_date",
    "Player",
    "Team",
    "Min",
    "Gls",
    "Ast",
    "shots",
    "shots_on_target",
    "tackles_won",
    "crosses",
    "interceptions",
    "fouls",
    "fouled",
    "offsides",
    "Pos",
]

fb_list = []

for season, teams in fbref_teams.items():
    for team in teams:
        path = os.path.join(
            BASE_DIR,
            season,
            "fbref",
            team,
            "match_reports",
            "master_player_stats.csv"
        )

        if os.path.exists(path):
            df = pd.read_csv(path, parse_dates=["match_date"])
            df = df[df["Player"] != "Matches"].copy()

            if "Min" in df.columns:
                df["Min"] = pd.to_numeric(
                    df["Min"].astype(str).str.replace(",", "", regex=False),
                    errors="coerce"
                )

            fb_list.append(df[[c for c in fb_cols if c in df.columns]])

if fb_list:
    fb = pd.concat(fb_list, ignore_index=True)

    fb["match_date"] = to_date(fb["match_date"])
    fb["_np"] = fb["Player"].map(norm)
    fb["_nt"] = fb["Team"].map(norm)

    fb = fb.rename(columns={
        "Min": "fb_min",
        "Gls": "fb_goals",
        "Ast": "fb_assists",
        "shots": "fb_shots",
        "shots_on_target": "fb_sot",
        "tackles_won": "fb_tackles_won",
        "crosses": "fb_crosses",
        "interceptions": "fb_interceptions",
        "fouls": "fb_fouls",
        "fouled": "fb_fouled",
        "offsides": "fb_offsides",
        "Pos": "fbref_position_raw",
    })

    fb = fb.drop(columns=["Player", "Team"])
    fb = fb.drop_duplicates(subset=["match_date", "_np", "_nt"])

    before = len(master)

    master = master.merge(
        fb,
        left_on=["date", "_np", "_nt"],
        right_on=["match_date", "_np", "_nt"],
        how="left",
        validate="many_to_one"
    )

    master = master.drop(columns=["match_date"])

    print("\n" + "=" * 80)
    print("STEP 4 — FBREF MATCH STATS")
    print("=" * 80)
    print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
    print("Row count preserved:", len(master) == before)
    print(f"Matched FBRef rows: {master['fb_min'].notna().sum():,}/{len(master):,}")

else:
    print("\nSTEP 4 — No FBRef files found")


# ============================================================
# STEP 5 — Squad Injury Burden
# Join key: season + date + team
# Premier League only
# ============================================================

team_injury_list = []

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"SEASON_{season_key}",
        "processed_injuries",
        f"team_match_injury_outcomes_{season}.csv"
    )

    if os.path.exists(path):
        df = pd.read_csv(path)
        team_injury_list.append(df)

if team_injury_list:
    it = pd.concat(team_injury_list, ignore_index=True)

    it["Fixture_Date"] = to_date(it["Fixture_Date"])
    it["_nt"] = it["Team"].map(norm)
    it["season_key"] = it["Season"].astype(str)

    it = it.rename(columns={
        "Players_Missing_Total": "squad_missing_total",
        "Players_Missing_Injury": "squad_injured_count",
        "Players_Missing_Non_Injury": "squad_non_injury_missing_count",
        "Soft_Tissue_Missing": "squad_soft_tissue_count",
        "Max_Reported_Days_Out": "squad_max_days_out",
        "Mean_Reported_Days_Out": "squad_avg_days_out",
    })

    it_slim = it[
        [
            "season_key",
            "Fixture_Date",
            "_nt",
            "squad_missing_total",
            "squad_injured_count",
            "squad_non_injury_missing_count",
            "squad_soft_tissue_count",
            "squad_max_days_out",
            "squad_avg_days_out",
        ]
    ].drop_duplicates(subset=["season_key", "Fixture_Date", "_nt"])

    before = len(master)

    master = master.merge(
        it_slim,
        left_on=["season_key", "date", "_nt"],
        right_on=["season_key", "Fixture_Date", "_nt"],
        how="left",
        validate="many_to_one"
    )

    master = master.drop(columns=["Fixture_Date"])

    print("\n" + "=" * 80)
    print("STEP 5 — SQUAD INJURY BURDEN")
    print("=" * 80)
    print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
    print("Row count preserved:", len(master) == before)
    print(f"Matched squad injury rows: {master['squad_injured_count'].notna().sum():,}/{len(master):,}")

else:
    print("\nSTEP 5 — No squad injury files found")


# ============================================================
# STEP 6 — Workload & Fatigue Features for ALL Rows
#
# Recomputed from:
#   date
#   minutes_played
#   is_home
#
# Full-coverage features:
#   rest_days
#   min_last_7d
#   min_last_28d
#   acwr_ratio
#   high_congestion_flag
#   consecutive_away_games
#   minutes_workload
# ============================================================

print("\n" + "=" * 80)
print("STEP 6 — WORKLOAD & FATIGUE FEATURES FOR ALL ROWS")
print("=" * 80)

master["date"] = pd.to_datetime(master["date"], errors="coerce")
master["minutes_workload"] = pd.to_numeric(master["minutes_played"], errors="coerce").fillna(0)

# Group by normalized player + team.
# This avoids problems if player_id is reused or missing.
master["_gk"] = master["_np"] + "||" + master["_nt"]

master = master.sort_values(["_gk", "date", "fixture_id"]).reset_index(drop=True)

n = len(master)

rest_days_arr = np.full(n, np.nan, dtype=float)
min_last_7d_arr = np.zeros(n, dtype=float)
min_last_28d_arr = np.zeros(n, dtype=float)
consec_away_arr = np.zeros(n, dtype=np.int16)

print(f"Computing workload features for {master['_gk'].nunique():,} player-team groups ({n:,} rows)...")

for gk, grp_idx in master.groupby("_gk", sort=False).groups.items():
    idx = list(grp_idx)

    dates = master.loc[idx, "date"].tolist()
    mins = master.loc[idx, "minutes_workload"].tolist()

    if "is_home" in master.columns:
        is_away = (~master.loc[idx, "is_home"].fillna(False).astype(bool)).tolist()
    else:
        is_away = [False] * len(idx)

    ng = len(idx)

    pfx = [0.0] * (ng + 1)
    for j, m in enumerate(mins):
        pfx[j + 1] = pfx[j] + float(m)

    consec = [0] * ng

    if ng > 0:
        consec[0] = 1 if is_away[0] else 0

    for j in range(1, ng):
        consec[j] = consec[j - 1] + 1 if is_away[j] else 0

    for i, gi in enumerate(idx):
        d = dates[i]

        if pd.isna(d):
            continue

        if i > 0 and pd.notna(dates[i - 1]):
            rest_days_arr[gi] = min(21.0, float((d - dates[i - 1]).days))

        hi = bisect_left(dates, d)
        lo7 = bisect_left(dates, d - pd.Timedelta(days=7))
        lo28 = bisect_left(dates, d - pd.Timedelta(days=28))

        min_last_7d_arr[gi] = pfx[hi] - pfx[lo7]
        min_last_28d_arr[gi] = pfx[hi] - pfx[lo28]
        consec_away_arr[gi] = consec[i]

chronic = min_last_28d_arr / 4.0

acwr = np.where(
    chronic > 0,
    min_last_7d_arr / chronic,
    0.0
)

acwr = np.clip(acwr, 0, 4)

high_congestion = np.where(
    np.isnan(rest_days_arr),
    0,
    (rest_days_arr <= 6).astype(int)
)

master["rest_days"] = rest_days_arr
master["min_last_7d"] = min_last_7d_arr
master["min_last_28d"] = min_last_28d_arr
master["acwr_ratio"] = np.round(acwr, 3)
master["high_congestion_flag"] = high_congestion.astype(int)
master["consecutive_away_games"] = consec_away_arr.astype(int)

master = master.drop(columns=["_gk"], errors="ignore")

print("Workload features recomputed.")

for col in [
    "rest_days",
    "min_last_7d",
    "min_last_28d",
    "acwr_ratio",
    "high_congestion_flag",
    "consecutive_away_games",
    "minutes_workload",
]:
    null_pct = master[col].isna().mean() * 100
    mean_val = master[col].mean()
    max_val = master[col].max()

    print(
        f"{col:<30} "
        f"null%: {null_pct:5.1f} | "
        f"mean: {mean_val:8.2f} | "
        f"max: {max_val:8.2f}"
    )


# ============================================================
# STEP 7 — Lag-based Individual Injury Features
#
# Direct same-day injury matching returns zero because:
#   injury table = missed fixtures
#   master table = played fixtures
#
# Correct features:
#   fixtures_missed_last_30d
#   fixtures_missed_last_90d
#   returning_from_injury
#   days_since_last_injury
#   last_injury_type
#   player-season injury burden
# ============================================================

print("\n" + "=" * 80)
print("STEP 7 — LAG-BASED INDIVIDUAL INJURY FEATURES")
print("=" * 80)

old_injury_cols = [
    "is_injured",
    "is_injured_on_matchday",
    "injury_days_out",
    "injury_type",
    "had_injury_absence_last_28d",
    "injury_absences_last_28d",
    "days_since_last_injury_absence",
    "last_injury_type",
    "fixtures_missed_last_30d",
    "fixtures_missed_last_90d",
    "returning_from_injury",
    "days_since_last_injury",
    "season_matches_missed_total",
    "season_matches_missed_injury",
    "season_main_reason",
    "season_main_injury_category",
    "season_max_reported_days_out",
    "has_player_season_injury_record",
]

master = master.drop(
    columns=[c for c in old_injury_cols if c in master.columns],
    errors="ignore"
)

# ------------------------------------------------------------
# STEP 7A — Load ALL_TEAMS physical absence records
# ------------------------------------------------------------

absence_list = []

for season in SEASONS:
    season_key = season.replace("-", "_")

    possible_paths = [
        os.path.join(
            BASE_DIR,
            f"SEASON_{season_key}",
            "injuries",
            f"ALL_TEAMS_{season}_injuries_days_out.csv"
        ),
        os.path.join(
            BASE_DIR,
            season,
            "injuries",
            f"ALL_TEAMS_{season}_injuries_days_out.csv"
        ),
    ]

    for path in possible_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["source_season"] = season
            absence_list.append(df)
            break

if absence_list:
    ia = pd.concat(absence_list, ignore_index=True)

    EXCLUDE_REASONS = {
        "Suspended",
        "Yellow Cards",
        "Red Card",
        "Inactive",
        "Lacking Match Fitness",
        "Rest",
        "Loan agreement",
        "Coach's decision",
        "Personal Reasons",
        "International duty",
        "National selection",
    }

    ia = ia[~ia["reason"].isin(EXCLUDE_REASONS)].copy()

    ia["fixture_date"] = to_date(ia["fixture_date"])
    ia["_np"] = ia["player_name"].map(norm)
    ia["_nt"] = ia["team_name"].map(norm)

    ia = ia.dropna(subset=["fixture_date"])
    ia = ia.drop_duplicates(subset=["_np", "_nt", "fixture_date"])

    print(f"Physical absence records: {len(ia):,}")

    if "reason" in ia.columns:
        print("\nTop absence reasons kept:")
        print(ia["reason"].value_counts().head(12))

    injury_lookup = {}

    for key, grp in ia.groupby(["_np", "_nt"]):
        grp = grp.sort_values("fixture_date")

        injury_lookup[key] = {
            "dates": grp["fixture_date"].tolist(),
            "types": grp["injury_type"].astype("object").tolist()
        }

    print(f"\nUnique player-team pairs with physical absence history: {len(injury_lookup):,}")

    missed_30 = np.zeros(len(master), dtype=np.int16)
    missed_90 = np.zeros(len(master), dtype=np.int16)
    returning = np.zeros(len(master), dtype=np.int8)
    days_since = np.full(len(master), np.nan, dtype=float)
    last_types = np.array([pd.NA] * len(master), dtype=object)

    for i, (d, np_, nt_) in enumerate(
        zip(master["date"].tolist(), master["_np"].tolist(), master["_nt"].tolist())
    ):
        item = injury_lookup.get((np_, nt_))

        if item is None or pd.isna(d):
            continue

        dates = item["dates"]
        types = item["types"]

        hi = bisect_left(dates, d)

        lo30 = bisect_left(dates, d - pd.Timedelta(days=30))
        lo90 = bisect_left(dates, d - pd.Timedelta(days=90))

        missed_30[i] = max(0, hi - lo30)
        missed_90[i] = max(0, hi - lo90)
        returning[i] = 1 if missed_30[i] > 0 else 0

        last_idx = hi - 1

        if last_idx >= 0:
            days_since[i] = (d - dates[last_idx]).days
            last_types[i] = types[last_idx]

    master["fixtures_missed_last_30d"] = missed_30
    master["fixtures_missed_last_90d"] = missed_90
    master["returning_from_injury"] = returning
    master["days_since_last_injury"] = days_since
    master["last_injury_type"] = last_types

    print("\nLag injury feature summary:")
    print("returning_from_injury rows:", int(master["returning_from_injury"].sum()))
    print("fixtures_missed_last_30d > 0:", int((master["fixtures_missed_last_30d"] > 0).sum()))
    print("fixtures_missed_last_90d > 0:", int((master["fixtures_missed_last_90d"] > 0).sum()))
    print("days_since_last_injury non-null:", int(master["days_since_last_injury"].notna().sum()))

else:
    master["fixtures_missed_last_30d"] = 0
    master["fixtures_missed_last_90d"] = 0
    master["returning_from_injury"] = 0
    master["days_since_last_injury"] = np.nan
    master["last_injury_type"] = pd.Series(pd.NA, index=master.index, dtype="object")

    print("No ALL_TEAMS injury absence files found")


# ------------------------------------------------------------
# STEP 7B — Player-season absence burden
# ------------------------------------------------------------

player_burden_list = []

for season in SEASONS:
    season_key = season.replace("-", "_")

    path = os.path.join(
        BASE_DIR,
        f"SEASON_{season_key}",
        "processed_injuries",
        f"player_season_absence_burden_{season}.csv"
    )

    if os.path.exists(path):
        df = pd.read_csv(path)
        df["source_season"] = season
        player_burden_list.append(df)

if player_burden_list:
    player_burden = pd.concat(player_burden_list, ignore_index=True)

    player_burden["season_key"] = player_burden["source_season"].astype(str)
    player_burden["_nt"] = player_burden["Team"].map(norm)
    player_burden["_np"] = player_burden["Player"].map(norm)

    player_burden_slim = player_burden[
        [
            "season_key",
            "_np",
            "_nt",
            "Matches_Missed_Total",
            "Matches_Missed_Injury",
            "Main_Reason",
            "Main_Injury_Category",
            "Max_Reported_Days_Out",
        ]
    ].copy()

    player_burden_slim = player_burden_slim.rename(columns={
        "Matches_Missed_Total": "season_matches_missed_total",
        "Matches_Missed_Injury": "season_matches_missed_injury",
        "Main_Reason": "season_main_reason",
        "Main_Injury_Category": "season_main_injury_category",
        "Max_Reported_Days_Out": "season_max_reported_days_out",
    })

    player_burden_slim = player_burden_slim.drop_duplicates(
        subset=["season_key", "_np", "_nt"]
    )

    before = len(master)

    master = master.merge(
        player_burden_slim,
        on=["season_key", "_np", "_nt"],
        how="left",
        validate="many_to_one"
    )

    master["has_player_season_injury_record"] = (
        master["season_matches_missed_injury"].notna()
    ).astype(int)

    print("\nPlayer-season injury burden")
    print("Row count preserved:", len(master) == before)
    print("Rows with season injury burden record:", int(master["has_player_season_injury_record"].sum()))

else:
    master["season_matches_missed_total"] = np.nan
    master["season_matches_missed_injury"] = np.nan
    master["season_main_reason"] = pd.Series(pd.NA, index=master.index, dtype="object")
    master["season_main_injury_category"] = pd.Series(pd.NA, index=master.index, dtype="object")
    master["season_max_reported_days_out"] = np.nan
    master["has_player_season_injury_record"] = 0

    print("\nNo player-season injury burden files found")

master["season_matches_missed_total"] = master["season_matches_missed_total"].fillna(0)
master["season_matches_missed_injury"] = master["season_matches_missed_injury"].fillna(0)
master["season_max_reported_days_out"] = master["season_max_reported_days_out"].fillna(0)


# ============================================================
# STEP 8 — Target Variables
#
# Primary targets:
#   next_api_rating
#   api_rating_decline_flag
#
# Secondary targets:
#   next_sofascore_rating
#   rating_decline_flag
#
# Important:
# API rating = 0 is treated as missing / sentinel.
# Grouping uses normalized player + team, not player_id.
# ============================================================

print("\n" + "=" * 80)
print("STEP 8 — TARGET VARIABLES")
print("=" * 80)

# Recreate helper group key if needed
if "_np" not in master.columns:
    master["_np"] = master["player_name"].map(norm)

if "_nt" not in master.columns:
    master["_nt"] = master["player_team"].map(norm)

master["_player_team_key"] = master["_np"] + "||" + master["_nt"]

# Sort chronologically inside player-team career
master = master.sort_values(
    ["_player_team_key", "date", "fixture_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# API rating targets
# ------------------------------------------------------------

master["api_rating_clean"] = pd.to_numeric(
    master["rating"],
    errors="coerce"
)

# rating = 0 is a sentinel, not a real performance rating
master.loc[master["api_rating_clean"] <= 0, "api_rating_clean"] = np.nan

master["next_api_rating"] = (
    master
    .groupby("_player_team_key")["api_rating_clean"]
    .shift(-1)
)

API_DECLINE_THRESHOLD = 0.5

master["api_rating_decline_flag"] = (
    master["api_rating_clean"].notna()
    & master["next_api_rating"].notna()
    & (
        master["next_api_rating"]
        < master["api_rating_clean"] - API_DECLINE_THRESHOLD
    )
).astype(int)

# ------------------------------------------------------------
# Optional SofaScore targets
# ------------------------------------------------------------

if "sofascore_rating" in master.columns:
    master["next_sofascore_rating"] = (
        master
        .groupby("_player_team_key")["sofascore_rating"]
        .shift(-1)
    )

    master["rating_decline_flag"] = (
        master["sofascore_rating"].notna()
        & master["next_sofascore_rating"].notna()
        & (
            master["next_sofascore_rating"]
            < master["sofascore_rating"] - 0.5
        )
    ).astype(int)

# ------------------------------------------------------------
# Target report
# ------------------------------------------------------------

print("API rating clean available:",
      f"{master['api_rating_clean'].notna().sum():,}/{len(master):,}",
      f"({master['api_rating_clean'].notna().mean() * 100:.1f}%)")

print("next_api_rating available:",
      f"{master['next_api_rating'].notna().sum():,}/{len(master):,}",
      f"({master['next_api_rating'].notna().mean() * 100:.1f}%)")

api_pos = int(master["api_rating_decline_flag"].sum())
api_neg = int((master["api_rating_decline_flag"] == 0).sum())

print("api_rating_decline_flag positives:",
      f"{api_pos:,}/{len(master):,}",
      f"({master['api_rating_decline_flag'].mean() * 100:.1f}%)")

if api_pos > 0:
    print("API scale_pos_weight:",
          round(api_neg / api_pos, 2))

if "next_sofascore_rating" in master.columns:
    print("\nSofaScore target available:",
          f"{master['next_sofascore_rating'].notna().sum():,}/{len(master):,}",
          f"({master['next_sofascore_rating'].notna().mean() * 100:.1f}%)")

    print("SofaScore decline positives:",
          f"{int(master['rating_decline_flag'].sum()):,}/{len(master):,}",
          f"({master['rating_decline_flag'].mean() * 100:.1f}%)")


# ============================================================
# STEP 9 — Clean helper columns
# ============================================================

master = master.drop(columns=["_np", "_nt"], errors="ignore")

print("\n" + "=" * 80)
print("FINAL MASTER TABLE BUILT")
print("=" * 80)

print("Shape:", master.shape)

print("\nRows by competition:")
print(master["competition"].value_counts())

print("\nRows by season:")
print(master["season_key"].value_counts().sort_index())

print("\nFeature fill-rate:")
for col, label in [
    ("rating", "API player rating"),
    ("goals_for", "API team results"),
    ("sofascore_rating", "SofaScore rating"),
    ("fb_min", "FBRef action stats"),
    ("squad_injured_count", "Squad injury burden"),
    ("rest_days", "Rest days"),
    ("min_last_7d", "Minutes last 7d"),
    ("min_last_28d", "Minutes last 28d"),
    ("acwr_ratio", "ACWR ratio"),
    ("high_congestion_flag", "High congestion flag"),
    ("consecutive_away_games", "Consecutive away games"),
    ("returning_from_injury", "Returning from injury"),
    ("fixtures_missed_last_30d", "Fixtures missed last 30d"),
    ("fixtures_missed_last_90d", "Fixtures missed last 90d"),
    ("days_since_last_injury", "Days since last injury"),
    ("has_player_season_injury_record", "Player-season injury record"),
    ("season_matches_missed_injury", "Season injury absences"),
    ("next_sofascore_rating", "Target: next SofaScore rating"),
    ("rating_decline_flag", "Target: decline flag"),
]:
    if col in master.columns:
        if col in [
            "returning_from_injury",
            "rating_decline_flag",
            "has_player_season_injury_record",
            "high_congestion_flag",
        ]:
            n = int(master[col].sum())
            pct = master[col].mean() * 100
        elif col in [
            "fixtures_missed_last_30d",
            "fixtures_missed_last_90d",
            "season_matches_missed_injury",
        ]:
            n = int((master[col] > 0).sum())
            pct = n / len(master) * 100
        else:
            n = int(master[col].notna().sum())
            pct = n / len(master) * 100

        print(f"{label:<40} {n:8,} ({pct:5.1f}%)")


# ============================================================
# STEP 10 — Save final CSV
# ============================================================

if master.shape[0] != 68722:
    raise ValueError(
        f"Row count changed to {master.shape[0]:,}. "
        "Expected 68,722. One of the joins may have duplicated rows."
    )

master.to_csv(OUT_PATH, index=False)

print("\n" + "=" * 80)
print("SAVED FINAL MASTER TABLE")
print("=" * 80)
print("Path:", OUT_PATH)
print("Shape:", master.shape)
print("File size MB:", round(os.path.getsize(OUT_PATH) / 1e6, 2))

STEP 1 — BASE: API PLAYER STATS
Shape: 68,722 rows × 37 cols

Rows by competition:
competition
Premier League      45552
League Cup          10997
FA Cup               6972
Champions League     5081
Community Shield      120
Name: count, dtype: int64

STEP 2 — API TEAM RESULTS
Shape: 68,722 rows × 54 cols
Row count preserved: True
Matched team rows: 68,722/68,722

STEP 3 — SOFASCORE DYNAMIC MASTER
Shape: 68,722 rows × 62 cols
Row count preserved: True
Matched SofaScore rows: 13,014/68,722

STEP 4 — FBREF MATCH STATS
Shape: 68,722 rows × 74 cols
Row count preserved: True
Matched FBRef rows: 7,156/68,722

STEP 5 — SQUAD INJURY BURDEN
Shape: 68,722 rows × 80 cols
Row count preserved: True
Matched squad injury rows: 45,552/68,722

STEP 6 — WORKLOAD & FATIGUE FEATURES FOR ALL ROWS
Computing workload features for 6,759 player-team groups (68,722 rows)...


/tmp/ipykernel_66190/659775152.py:563: RuntimeWarning: invalid value encountered in divide
  min_last_7d_arr / chronic,


Workload features recomputed.
rest_days                      null%:   9.8 | mean:     8.77 | max:    21.00
min_last_7d                    null%:   0.0 | mean:    35.54 | max:   215.00
min_last_28d                   null%:   0.0 | mean:   145.40 | max:   750.00
acwr_ratio                     null%:   0.0 | mean:     0.75 | max:     4.00
high_congestion_flag           null%:   0.0 | mean:     0.42 | max:     1.00
consecutive_away_games         null%:   0.0 | mean:     0.68 | max:     7.00
minutes_workload               null%:   0.0 | mean:    49.80 | max:   120.00

STEP 7 — LAG-BASED INDIVIDUAL INJURY FEATURES
Physical absence records: 8,512

Top absence reasons kept:
reason
Knee Injury               2232
Thigh Injury              1421
Muscle Injury              924
Ankle Injury               908
Injury                     653
Calf Injury                524
Hamstring Injury           232
Groin Injury               206
Knock                      169
Shoulder Injury            156
Achilles

## Section 9e — Starter vs Substitute Bias in Target Variables

### The problem

The `api_rating_decline_flag` is **confounded by role changes**. Substitutes receive a structurally lower rating than starters because they play fewer minutes:

| Role | Mean rating | Std | Mean minutes |
|------|------------|-----|-------------|
| Starter (`is_substitute=0`) | **6.99** | 0.61 | 82.9 min |
| Substitute (`is_substitute=1`) | **6.66** | 0.39 | 9.5 min |

When the *next* match has a different role, the flag is measuring **role demotion**, not pure performance decline:

| Transition (current→next) | Rows | Decline flags | Decline rate |
|---|---|---|---|
| Starter → Starter | 26,051 | 6,540 | 25.1% — genuine signal |
| Starter → Sub | 4,778 | 1,511 | **31.6%** — inflated: fewer minutes = lower rating by construction |
| Sub → Sub | 4,140 | 573 | 13.8% — noisy: two short appearances |
| Sub → Starter | 5,132 | 519 | 10.1% — mostly improvements, minority mislabelled as decline |

### Fix applied in this section

1. **`api_rating_decline_flag` updated** — only fires when the player played ≥ `MIN_MINUTES` (default 45) in **both** the current and next match. This compares like-for-like meaningful appearances only.
2. **Two new context features added** — `next_is_substitute` and `next_minutes_played` so the model can learn role-change patterns even where the flag is suppressed.

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9e — Starter vs Substitute Bias Correction
#
# Problem:
# API rating is structurally lower for substitutes because they play fewer minutes.
# Comparing a starter's current rating to a substitute's next-match rating may
# label a role demotion as a performance decline.
#
# Fix:
#   1. Add next_is_substitute and next_minutes_played.
#   2. Recompute api_rating_decline_flag only when BOTH current and next match
#      have minutes_played >= MIN_MINUTES.
# ─────────────────────────────────────────────────────────────────────────────

import os
import unicodedata
import numpy as np
import pandas as pd

OUT_PATH = "./Fixture_IQ_Data_Seasons_2022-2025.csv"

DECLINE_THRESHOLD = 0.5
MIN_MINUTES = 45


def norm(s):
    if pd.isna(s):
        return ""

    s = unicodedata.normalize("NFD", str(s).lower().strip())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")

    return (
        s.replace("'", "")
         .replace("-", " ")
         .replace(".", "")
         .replace("_", " ")
         .replace("&", "and")
         .strip()
    )


# ── 1. Load & sort ───────────────────────────────────────────────────────────

master = pd.read_csv(OUT_PATH)

master["date"] = pd.to_datetime(master["date"], errors="coerce")
master["_gk"] = (
    master["player_name"].map(norm)
    + "||"
    + master["player_team"].map(norm)
)

master = master.sort_values(["_gk", "date", "fixture_id"]).reset_index(drop=True)


# ── 2. Clean API rating ──────────────────────────────────────────────────────

master["api_rating_clean"] = pd.to_numeric(master["rating"], errors="coerce")
master.loc[master["api_rating_clean"] <= 0, "api_rating_clean"] = np.nan


# ── 3. Add next-match context features ───────────────────────────────────────

master["next_api_rating"] = (
    master.groupby("_gk")["api_rating_clean"].shift(-1)
)

master["next_is_substitute"] = (
    master.groupby("_gk")["is_substitute"].shift(-1)
)

master["next_minutes_played"] = (
    master.groupby("_gk")["minutes_played"].shift(-1)
)


# ── 4. Show rating distribution by role ──────────────────────────────────────

r = master[master["api_rating_clean"].notna()].copy()

print("=" * 80)
print("API RATING BY ROLE")
print("=" * 80)

stats = (
    r.groupby("is_substitute")["api_rating_clean"]
     .agg(["count", "mean", "std", "min", "max"])
)

stats.index = [
    "Starter (is_substitute=0)" if i == 0 else "Substitute (is_substitute=1)"
    for i in stats.index
]

print(stats.round(3).to_string())

print("\n" + "=" * 80)
print("MINUTES PLAYED BY ROLE")
print("=" * 80)

mstats = (
    master.groupby("is_substitute")["minutes_played"]
    .agg(["count", "mean", "std", "min", "max"])
)

mstats.index = [
    "Starter" if i == 0 else "Substitute"
    for i in mstats.index
]

print(mstats.round(1).to_string())


# ── 5. Before-fix flag for comparison ────────────────────────────────────────

master["api_rating_decline_flag_before_min_guard"] = (
    master["api_rating_clean"].notna()
    & master["next_api_rating"].notna()
    & (
        master["next_api_rating"]
        < master["api_rating_clean"] - DECLINE_THRESHOLD
    )
).astype(int)


# ── 6. Role-transition analysis before fix ───────────────────────────────────

print("\n" + "=" * 80)
print("DECLINE FLAGS BY ROLE TRANSITION — BEFORE MINUTES GUARD")
print("=" * 80)

has_both = (
    master["api_rating_clean"].notna()
    & master["next_api_rating"].notna()
    & master["next_is_substitute"].notna()
)

trans_df = master[has_both].copy()

trans_df["next_sub_int"] = trans_df["next_is_substitute"].astype(int)

trans_df["transition"] = (
    trans_df["is_substitute"].astype(int).astype(str)
    + "→"
    + trans_df["next_sub_int"].astype(str)
).map({
    "0→0": "Starter→Starter",
    "1→0": "Sub→Starter",
    "0→1": "Starter→Sub",
    "1→1": "Sub→Sub",
}).fillna("unknown")

grp_before = (
    trans_df
    .groupby("transition")
    .agg(
        rows=("api_rating_decline_flag_before_min_guard", "count"),
        declines=("api_rating_decline_flag_before_min_guard", "sum"),
    )
)

grp_before["decline_rate"] = (
    grp_before["declines"] / grp_before["rows"] * 100
)

print(grp_before.round(2).to_string())


# ── 7. Apply minutes guard ───────────────────────────────────────────────────

current_played_enough = master["minutes_played"] >= MIN_MINUTES
next_played_enough = master["next_minutes_played"].fillna(0) >= MIN_MINUTES

master["api_rating_decline_flag"] = (
    current_played_enough
    & next_played_enough
    & master["api_rating_clean"].notna()
    & master["next_api_rating"].notna()
    & (
        master["next_api_rating"]
        < master["api_rating_clean"] - DECLINE_THRESHOLD
    )
).astype(int)


# ── 8. Role-transition analysis after fix ────────────────────────────────────

print("\n" + "=" * 80)
print(f"DECLINE FLAGS BY ROLE TRANSITION — AFTER MINUTES GUARD ≥ {MIN_MINUTES} MIN")
print("=" * 80)

trans_df_after = master.loc[trans_df.index].copy()
trans_df_after["transition"] = trans_df["transition"].values

grp_after = (
    trans_df_after
    .groupby("transition")
    .agg(
        rows=("api_rating_decline_flag", "count"),
        declines=("api_rating_decline_flag", "sum"),
    )
)

grp_after["decline_rate"] = (
    grp_after["declines"] / grp_after["rows"] * 100
)

print(grp_after.round(2).to_string())


# ── 9. Summary: flag change impact ───────────────────────────────────────────

old_flag = master["api_rating_decline_flag_before_min_guard"]
new_flag = master["api_rating_decline_flag"]

removed = int(((old_flag == 1) & (new_flag == 0)).sum())
retained = int(((old_flag == 1) & (new_flag == 1)).sum())

old_rate = old_flag.mean() * 100
new_rate = new_flag.mean() * 100

print("\n" + "=" * 80)
print("FLAG CHANGE IMPACT")
print("=" * 80)
print(f"Decline threshold: {DECLINE_THRESHOLD}")
print(f"Minimum minutes in current and next match: {MIN_MINUTES}")
print(f"Flags retained: {retained:,}")
print(f"Flags removed:  {removed:,}")
print(f"Old positive rate: {old_rate:.1f}%")
print(f"New positive rate: {new_rate:.1f}%")

if new_flag.sum() > 0:
    new_spw = (len(master) - new_flag.sum()) / new_flag.sum()
    print(f"New scale_pos_weight: {new_spw:.1f}")


# ── 10. Save updated master ──────────────────────────────────────────────────

master = master.drop(columns=["_gk"], errors="ignore")

master = master.sort_values(["player_name", "player_team", "date", "fixture_id"]).reset_index(drop=True)

master.to_csv(OUT_PATH, index=False)

print("\n" + "=" * 80)
print("MASTER TABLE UPDATED WITH MINUTES-GUARDED API TARGET")
print("=" * 80)
print("Path:", OUT_PATH)
print("Shape:", master.shape)
print("File size MB:", round(os.path.getsize(OUT_PATH) / 1e6, 2))
print("New columns: next_is_substitute, next_minutes_played")
print("Updated: api_rating_decline_flag")


/tmp/ipykernel_66190/3568698578.py:45: DtypeWarning: Columns (89) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv(OUT_PATH)


API RATING BY ROLE
                              count   mean    std  min   max
Starter (is_substitute=0)     37715  6.989  0.610  3.0  10.0
Substitute (is_substitute=1)  13173  6.662  0.389  3.6   9.5

MINUTES PLAYED BY ROLE
            count  mean   std  min  max
Starter     37730  82.9  13.4    4  120
Substitute  30992   9.5  13.8    0   92

DECLINE FLAGS BY ROLE TRANSITION — BEFORE MINUTES GUARD
                  rows  declines  decline_rate
transition                                    
Starter→Starter  26144      6561         25.10
Starter→Sub       4754      1499         31.53
Sub→Starter       5122       518         10.11
Sub→Sub           4146       576         13.89

DECLINE FLAGS BY ROLE TRANSITION — AFTER MINUTES GUARD ≥ 45 MIN
                  rows  declines  decline_rate
transition                                    
Starter→Starter  26144      6421         24.56
Starter→Sub       4754       172          3.62
Sub→Starter       5122       151          2.95
Sub→Sub        

In [26]:
import pandas as pd

OUT_PATH = "./Fixture_IQ_Data_Seasons_2022-2025.csv"

df = pd.read_csv(OUT_PATH, low_memory=False)

summary = pd.DataFrame({
    "column_number": range(1, len(df.columns) + 1),
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "null_pct": (df.isna().mean() * 100).round(2).values,
    "non_null": df.notna().sum().values,
})

print("Shape:", df.shape)
display(summary)

print("\nImportant target columns:")
for col in [
    "next_api_rating",
    "api_rating_decline_flag",
    "next_sofascore_rating",
    "rating_decline_flag",
    "next_is_substitute",
    "next_minutes_played",
]:
    if col in df.columns:
        if "flag" in col:
            print(col, "positive:", int(df[col].sum()), f"({df[col].mean()*100:.2f}%)")
        else:
            print(col, "non-null:", df[col].notna().sum(), f"({df[col].notna().mean()*100:.2f}%)")

Shape: (68722, 105)


,column_number,column,dtype,null_pct,non_null
0,1,fixture_id,int64,0.00,68722
1,2,date,object,0.00,68722
2,3,competition,object,0.00,68722
3,4,season,int64,0.00,68722
4,5,round,object,0.00,68722
5,6,home_team,object,0.00,68722
6,7,away_team,object,0.00,68722
7,8,player_team,object,0.00,68722
8,9,player_id,int64,0.00,68722
9,10,player_name,object,0.00,68722



Important target columns:
next_api_rating non-null: 46287 (67.35%)
api_rating_decline_flag positive: 6753 (9.83%)
next_sofascore_rating non-null: 12434 (18.09%)
rating_decline_flag positive: 1659 (2.41%)
next_is_substitute non-null: 61963 (90.16%)
next_minutes_played non-null: 61963 (90.16%)


## Section 10 — Master Table: Complete Column Reference

### `Fixture_IQ_Data_Seasons_2022-2025.csv` — 68,722 rows × 105 columns

Each row is **one player in one match**. There are no aggregate rows — every entry is a single player appearance in a specific fixture.

Columns are grouped by where the data came from and what each variable represents.

---

## GROUP 1 — Match Identity (cols 1–8)

*What match is this row about?*

| # | Column        | Type       | Null% | Example         | Meaning                                                                                                                   |
| - | ------------- | ---------- | ----- | --------------- | ------------------------------------------------------------------------------------------------------------------------- |
| 1 | `fixture_id`  | int        | 0%    | 1066064         | Unique numeric ID assigned by the API to each individual match. Useful for fixture-level joins.                           |
| 2 | `date`        | str / date | 0%    | 2024-08-13      | Calendar date the match was played. Used for rest windows, rolling workloads, lag injury features, and time-based splits. |
| 3 | `competition` | str        | 0%    | Premier League  | Competition name: `Premier League`, `Champions League`, `FA Cup`, `League Cup`, `Community Shield`.                       |
| 4 | `season`      | int        | 0%    | 2024            | Start year of the season. Example: `2024` means 2024–2025.                                                                |
| 5 | `round`       | str        | 0%    | Matchweek 1     | Competition round label. Highly variable across competitions, so encode carefully or drop.                                |
| 6 | `home_team`   | str        | 0%    | Manchester City | Full name of the home team.                                                                                               |
| 7 | `away_team`   | str        | 0%    | Arsenal         | Full name of the away team.                                                                                               |
| 8 | `player_team` | str        | 0%    | Arsenal         | Team this player appeared for. Useful for player-centric filtering and joins.                                             |

---

## GROUP 2 — Player Identity (cols 9–12)

*Who is this player?*

| #  | Column            | Type | Null% | Example        | Meaning                                                                                                                                                                                   |
| -- | ----------------- | ---- | ----- | -------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 9  | `player_id`       | int  | 0%    | 504648         | Numeric player ID from the API. **Important caveat**: many unmatched players share `player_id = 0`, so do not use alone as a grouping key. Prefer normalized `player_name + player_team`. |
| 10 | `player_name`     | str  | 0%    | Bernardo Silva | Player display name as returned by the API.                                                                                                                                               |
| 11 | `player_number`   | int  | 0%    | 20             | Shirt number in that specific match. Can change across seasons or teams.                                                                                                                  |
| 12 | `player_position` | str  | 0%    | M              | Broad API position code: **G**, **D**, **M**, **F**.                                                                                                                                      |

---

## GROUP 3 — Individual In-Match Performance — API Source (cols 13–34)

*What did this specific player do during the match?*

**Source**: API multi-competition player stats.
**Coverage**: 100% for all rows.

| #  | Column                  | Type  | Null% | Range | Meaning                                                                                                                                                      |
| -- | ----------------------- | ----- | ----- | ----- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| 13 | `minutes_played`        | int   | 0%    | 0–120 | Minutes played. Includes extra time. 0 means listed but did not play or no meaningful minutes.                                                               |
| 14 | `rating`                | float | 0%    | 0–10  | API match rating. **0 is a sentinel value**, not a real rating. Treat `rating = 0` as missing when modeling ratings.                                         |
| 15 | `is_captain`            | bool  | 0%    | 0/1   | Whether the player was captain.                                                                                                                              |
| 16 | `is_substitute`         | bool  | 0%    | 0/1   | Whether the player appeared as a substitute. Important because substitute ratings are structurally lower due to fewer minutes.                               |
| 17 | `shots_total`           | int   | 0%    | 0–9   | Total shots attempted.                                                                                                                                       |
| 18 | `shots_on_target`       | int   | 0%    | 0–8   | Shots on target.                                                                                                                                             |
| 19 | `goals`                 | int   | 0%    | 0–5   | Goals scored.                                                                                                                                                |
| 20 | `assists`               | int   | 0%    | 0–4   | Assists credited.                                                                                                                                            |
| 21 | `passes_total`          | int   | 0%    | 0–188 | Total passes attempted.                                                                                                                                      |
| 22 | `passes_key`            | int   | 0%    | 0–10  | Key passes creating scoring opportunities.                                                                                                                   |
| 23 | `passes_accuracy`       | float | 0%    | 0–181 | Passing accuracy field. **Data quality caveat**: values above 100 suggest some rows may contain counts rather than percentages. Clip or validate before use. |
| 24 | `dribbles_attempts`     | int   | 0%    | 0–20  | Dribble attempts.                                                                                                                                            |
| 25 | `dribbles_success`      | int   | 0%    | 0–15  | Successful dribbles.                                                                                                                                         |
| 26 | `tackles_total`         | int   | 0%    | 0–12  | Total tackles.                                                                                                                                               |
| 27 | `tackles_blocks`        | int   | 0%    | 0–7   | Blocks recorded in the tackle/block API category.                                                                                                            |
| 28 | `tackles_interceptions` | int   | 0%    | 0–13  | Interceptions.                                                                                                                                               |
| 29 | `duels_total`           | int   | 0%    | 0–45  | Total duels contested.                                                                                                                                       |
| 30 | `duels_won`             | int   | 0%    | 0–22  | Duels won.                                                                                                                                                   |
| 31 | `fouls_drawn`           | int   | 0%    | 0–8   | Fouls won.                                                                                                                                                   |
| 32 | `fouls_committed`       | int   | 0%    | 0–8   | Fouls committed.                                                                                                                                             |
| 33 | `cards_yellow`          | int   | 0%    | 0–2   | Yellow cards.                                                                                                                                                |
| 34 | `cards_red`             | int   | 0%    | 0–1   | Red cards.                                                                                                                                                   |

---

## GROUP 4 — Season and Team Match Context (cols 35–52)

*What was the match situation for this player’s team?*

**Source**: API team results.
**Join key**: `fixture_id + player_team`.

| #  | Column                | Type  | Null%  | Range         | Meaning                                                                                                              |
| -- | --------------------- | ----- | ------ | ------------- | -------------------------------------------------------------------------------------------------------------------- |
| 35 | `season_key`          | str   | 0%     | 2024-2025     | Clean season label. Easier to use than numeric `season`.                                                             |
| 36 | `is_home`             | bool  | 0%     | 0/1           | 1 if the player’s team was the home team.                                                                            |
| 37 | `opponent_team`       | str   | 0%     | —             | Opponent from the player team’s perspective.                                                                         |
| 38 | `goals_for`           | int   | 0%     | 0–9           | Goals scored by the player’s team.                                                                                   |
| 39 | `goals_against`       | int   | 0%     | 0–9           | Goals conceded by the player’s team.                                                                                 |
| 40 | `result`              | str   | 0%     | Win/Draw/Loss | Match result from the player team’s perspective.                                                                     |
| 41 | `points`              | int   | 0%     | 0/1/3         | Points earned from the player team’s perspective.                                                                    |
| 42 | `team_shots_on_goal`  | float | 22.10% | —             | Team shots on goal. Missing mainly for competitions/sources where detailed team stats were unavailable.              |
| 43 | `team_total_shots`    | float | 22.10% | —             | Team total shots.                                                                                                    |
| 44 | `team_possession`     | float | 22.10% | 0–100         | Team possession percentage.                                                                                          |
| 45 | `team_corner_kicks`   | float | 22.16% | —             | Team corners.                                                                                                        |
| 46 | `team_fouls`          | float | 22.10% | —             | Team fouls.                                                                                                          |
| 47 | `team_gk_saves`       | float | 22.33% | —             | Goalkeeper saves by player’s team.                                                                                   |
| 48 | `team_expected_goals` | float | 60.81% | —             | Team xG where available. Sparse across competitions.                                                                 |
| 49 | `opp_shots_on_goal`   | float | 100%   | —             | **Empty column. Drop before training.**                                                                              |
| 50 | `opp_total_shots`     | float | 100%   | —             | **Empty column. Drop before training.**                                                                              |
| 51 | `opp_possession`      | float | 100%   | —             | **Empty column. Drop before training.** Can be approximated as `100 - team_possession` where team possession exists. |
| 52 | `opp_expected_goals`  | float | 100%   | —             | **Empty column. Drop before training.**                                                                              |

---

## GROUP 5 — SofaScore Features Preserved from Source (cols 53–60)

*Sparse SofaScore variables, mainly useful for Premier League validation.*

**Source**: SofaScore dynamic master.
**Coverage**: mostly Premier League only.

| #  | Column                             | Type  | Null%  | Range | Meaning                                                                                                |
| -- | ---------------------------------- | ----- | ------ | ----- | ------------------------------------------------------------------------------------------------------ |
| 53 | `ss_minutes`                       | float | 73.73% | 0–120 | SofaScore minutes where matched. Sparse. Use `minutes_played` or `minutes_workload` for full coverage. |
| 54 | `sofascore_rating`                 | float | 81.06% | 3–10  | SofaScore player rating. Useful for PL-only validation, not the primary target.                        |
| 55 | `sofascore_rest_days`              | float | 73.73% | 0–21  | Original SofaScore rest-days feature. Preserved for comparison.                                        |
| 56 | `sofascore_high_congestion_flag`   | float | 73.73% | 0/1   | Original SofaScore congestion flag. Preserved for comparison.                                          |
| 57 | `sofascore_min_last_7d`            | float | 73.73% | —     | Original SofaScore 7-day workload.                                                                     |
| 58 | `sofascore_min_last_28d`           | float | 73.73% | —     | Original SofaScore 28-day workload.                                                                    |
| 59 | `sofascore_acwr_ratio`             | float | 73.73% | —     | Original SofaScore ACWR.                                                                               |
| 60 | `sofascore_consecutive_away_games` | float | 73.73% | —     | Original SofaScore consecutive-away feature.                                                           |

> **Training note**: These columns are sparse. Prefer the recomputed full-coverage workload features in cols 79–85.

---

## GROUP 6 — FBRef Detailed Action Stats (cols 61–72)

*Granular per-action statistics from FBRef match reports.*

**Source**: FBRef scraped match reports.
**Coverage**: selected teams only, around 10%.

| #  | Column               | Type   | Null%  | Range | Meaning                                                 |
| -- | -------------------- | ------ | ------ | ----- | ------------------------------------------------------- |
| 61 | `fb_min`             | float  | 89.59% | 1–120 | Minutes according to FBRef. Use for consistency checks. |
| 62 | `fb_goals`           | float  | 89.59% | 0–5   | Goals from FBRef.                                       |
| 63 | `fb_assists`         | float  | 89.59% | 0–4   | Assists from FBRef.                                     |
| 64 | `fb_shots`           | float  | 89.67% | 0–10  | Total shots from FBRef.                                 |
| 65 | `fb_sot`             | float  | 89.67% | 0–8   | Shots on target from FBRef.                             |
| 66 | `fb_tackles_won`     | float  | 89.67% | 0–8   | Successful tackles from FBRef.                          |
| 67 | `fb_crosses`         | float  | 89.67% | 0–22  | Crosses.                                                |
| 68 | `fb_interceptions`   | float  | 89.67% | 0–7   | Interceptions.                                          |
| 69 | `fb_fouls`           | float  | 89.67% | 0–7   | Fouls committed.                                        |
| 70 | `fb_fouled`          | float  | 89.67% | 0–7   | Fouls drawn.                                            |
| 71 | `fb_offsides`        | float  | 89.71% | 0–5   | Offsides.                                               |
| 72 | `fbref_position_raw` | object | 89.59% | —     | Raw FBRef position string.                              |

> **Training note**: Because these are ~90% null, either impute carefully, use missingness flags, or train a richer submodel on the FBRef-covered subset.

---

## GROUP 7 — Squad Injury Context (cols 73–78)

*How many players from the same squad were missing around this match?*

**Source**: `team_match_injury_outcomes`.
**Join key**: `season_key + date + player_team`.
**Coverage**: Premier League only, 45,552 rows.

| #  | Column                           | Type  | Null%  | Range | Meaning                                                |
| -- | -------------------------------- | ----- | ------ | ----- | ------------------------------------------------------ |
| 73 | `squad_missing_total`            | float | 33.72% | —     | Total squad players missing.                           |
| 74 | `squad_injured_count`            | float | 33.72% | 0–12  | Players missing due to injury/illness.                 |
| 75 | `squad_non_injury_missing_count` | float | 33.72% | —     | Players missing for non-injury reasons.                |
| 76 | `squad_soft_tissue_count`        | float | 33.72% | 0–7   | Soft-tissue injury burden.                             |
| 77 | `squad_max_days_out`             | float | 33.72% | —     | Maximum reported days out among missing squad players. |
| 78 | `squad_avg_days_out`             | float | 33.72% | —     | Average reported days out among missing squad players. |

---

## GROUP 8 — Recomputed Workload & Fatigue Features (cols 79–85)

*How fresh or fatigued is this player heading into this match?*

**Source**: Recomputed in integrated Section 9 from `date + minutes_played + is_home`.
**Coverage**: full or near-full across all competitions.

| #  | Column                   | Type  | Null% | Range | Meaning                                                                                                                       |
| -- | ------------------------ | ----- | ----- | ----- | ----------------------------------------------------------------------------------------------------------------------------- |
| 79 | `minutes_workload`       | int   | 0%    | 0–120 | Minutes used for workload calculations. Derived from API `minutes_played`.                                                    |
| 80 | `rest_days`              | float | 9.84% | 0–21  | Days since the same player-team’s previous appearance. Clipped at 21. Null for first appearance in each player-team timeline. |
| 81 | `min_last_7d`            | float | 0%    | 0–215 | Minutes played in the 7 days before this match. Acute workload.                                                               |
| 82 | `min_last_28d`           | float | 0%    | 0–750 | Minutes played in the 28 days before this match. Chronic workload window.                                                     |
| 83 | `acwr_ratio`             | float | 0%    | 0–4   | Acute:chronic workload ratio = `min_last_7d / (min_last_28d / 4)`, clipped at 4.                                              |
| 84 | `high_congestion_flag`   | int   | 0%    | 0/1   | 1 if `rest_days <= 6`. First appearances are assigned 0. Positive in 28,852 rows, around 42.0%.                               |
| 85 | `consecutive_away_games` | int   | 0%    | 0–7   | Consecutive away appearances up to and including this match. Resets to 0 on home appearances.                                 |

---

## GROUP 9 — Lag-Based Individual Injury Features (cols 86–96)

*Was this player recently returning from injury?*

**Source**: Integrated lag lookback over `ALL_TEAMS_injuries_days_out` plus player-season absence burden.
**Key idea**: injury absence records describe fixtures missed, while this master table contains played appearances. Therefore same-day injury matching is not the right feature; lag lookback is used instead.

| #  | Column                            | Type   | Null%  | Range | Meaning                                                                                                              |
| -- | --------------------------------- | ------ | ------ | ----- | -------------------------------------------------------------------------------------------------------------------- |
| 86 | `fixtures_missed_last_30d`        | int    | 0%     | 0–7   | Number of physical-injury/illness absences in the 30 days before this appearance. Positive in 231 rows, around 0.3%. |
| 87 | `fixtures_missed_last_90d`        | int    | 0%     | 0–14  | Same as above, but with a 90-day window. Positive in 602 rows, around 0.9%.                                          |
| 88 | `returning_from_injury`           | int    | 0%     | 0/1   | 1 if `fixtures_missed_last_30d > 0`.                                                                                 |
| 89 | `days_since_last_injury`          | float  | 98.23% | —     | Days since most recent injury absence. Sparse: 1,216 non-null rows. Usually drop or use with a missingness flag.     |
| 90 | `last_injury_type`                | object | 98.23% | —     | Injury type from most recent previous absence. Very sparse.                                                          |
| 91 | `season_matches_missed_total`     | float  | 0%     | —     | Player-season total missed matches. Filled with 0 when no record exists.                                             |
| 92 | `season_matches_missed_injury`    | float  | 0%     | —     | Player-season injury-related missed matches. Filled with 0 when no record exists. Positive in 1,577 rows.            |
| 93 | `season_main_reason`              | object | 97.36% | —     | Main absence reason for that player-season. Sparse.                                                                  |
| 94 | `season_main_injury_category`     | object | 97.36% | —     | Main injury category for that player-season. Sparse.                                                                 |
| 95 | `season_max_reported_days_out`    | float  | 0%     | —     | Maximum reported days out for that player-season. Filled with 0 when no record exists.                               |
| 96 | `has_player_season_injury_record` | int    | 0%     | 0/1   | 1 if the player-season had an injury/absence burden record. Positive in 1,817 rows.                                  |

---

## GROUP 10 — Helper and API Rating Target Columns (cols 97–100)

*Columns created during target engineering.*

| #   | Column                    | Type   | Null%  | Range | Meaning                                                                                                                                                                                 |
| --- | ------------------------- | ------ | ------ | ----- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 97  | `_player_team_key`        | object | 0%     | —     | Helper key: normalized `player_name + player_team`. Useful for debugging. **Drop before final modeling.**                                                                               |
| 98  | `api_rating_clean`        | float  | 25.95% | 3–10  | API rating with sentinel `rating = 0` converted to null. Use this instead of raw `rating` for rating-based modeling.                                                                    |
| 99  | `next_api_rating`         | float  | 32.65% | 3–10  | API rating in the same player-team’s next match. **Primary regression target.** Non-null in 46,287 rows, 67.35%.                                                                        |
| 100 | `api_rating_decline_flag` | int    | 0%     | 0/1   | **Primary classification target.** 1 if next API rating declines by more than 0.5, with both current and next appearances requiring at least 45 minutes. Positive in 6,753 rows, 9.83%. |

---

## GROUP 11 — Legacy SofaScore Targets (cols 101–102)

*Older targets based on SofaScore ratings.*

| #   | Column                  | Type  | Null%  | Range | Meaning                                                                                                                                            |
| --- | ----------------------- | ----- | ------ | ----- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| 101 | `next_sofascore_rating` | float | 81.91% | 3–10  | SofaScore rating in the next matched SofaScore appearance. Non-null in 12,434 rows, 18.09%. Use only for PL-only validation or secondary analysis. |
| 102 | `rating_decline_flag`   | int   | 0%     | 0/1   | Legacy SofaScore decline flag. Positive in 1,659 rows, 2.41%. Not recommended as primary target because non-PL rows are effectively negatives.     |

---

## GROUP 12 — Next-Match Role Context and Diagnostics (cols 103–105)

*Columns added during Section 9e to diagnose and correct starter/substitute bias.*

| #   | Column                                     | Type          | Null% | Range | Meaning                                                                                                                                                                                                    |
| --- | ------------------------------------------ | ------------- | ----- | ----- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 103 | `next_is_substitute`                       | object / bool | 9.84% | 0/1   | Whether the player was a substitute in their next match. Non-null in 61,963 rows, 90.16%. Useful for role-change analysis. **Do not use as a normal pre-match predictor unless the next lineup is known.** |
| 104 | `next_minutes_played`                      | float         | 9.84% | 0–120 | Minutes played in the next match. Used internally to enforce the 45-minute guard. **Future information; do not use as a normal pre-match predictor.**                                                      |
| 105 | `api_rating_decline_flag_before_min_guard` | int           | 0%    | 0/1   | Diagnostic version of the API decline target before the starter/substitute minutes guard. Useful for auditing. **Drop before final modeling.**                                                             |

---

## Target Variable Summary

| Target                                     | Type           | Coverage / Positive Rate     | Recommendation                            |
| ------------------------------------------ | -------------- | ---------------------------- | ----------------------------------------- |
| `next_api_rating`                          | Regression     | 46,287 non-null rows, 67.35% | **Primary regression target**             |
| `api_rating_decline_flag`                  | Classification | 6,753 positives, 9.83%       | **Primary classification target**         |
| `next_sofascore_rating`                    | Regression     | 12,434 non-null rows, 18.09% | Secondary / PL-only validation            |
| `rating_decline_flag`                      | Classification | 1,659 positives, 2.41%       | Legacy target; not recommended as primary |
| `api_rating_decline_flag_before_min_guard` | Diagnostic     | 100% filled                  | Audit only; drop before final modeling    |

---

## Important Modeling Notes

### 1. Recommended primary targets

Use:

```text
Regression target:
next_api_rating

Classification target:
api_rating_decline_flag
```

The API target covers all competitions and avoids the severe SofaScore coverage problem.

---

### 2. Rating sentinel handling

The raw API `rating` column has 0 values that should be treated as missing. Use:

```text
api_rating_clean
```

for modeling rather than raw `rating` if the rating itself is a feature.

---

### 3. Starter/substitute correction

The primary classification target has been corrected for starter/substitute bias:

```text
api_rating_decline_flag = 1 only if:
- current match minutes_played >= 45
- next match next_minutes_played >= 45
- next_api_rating < api_rating_clean - 0.5
```

This prevents role demotion or very short substitute appearances from being mislabeled as performance decline.

---

### 4. Next-match context columns are future information

These columns are useful for diagnostics and role-change analysis:

```text
next_is_substitute
next_minutes_played
```

But they are **future information**. Do not use them as regular pre-match predictors unless the modeling task explicitly assumes the next lineup/minutes are already known.

---

### 5. Columns to drop before standard pre-match modeling

Recommended drop list:

```text
fixture_id
round
player_id
player_number
opp_shots_on_goal
opp_total_shots
opp_possession
opp_expected_goals
_player_team_key
api_rating_decline_flag_before_min_guard
next_is_substitute
next_minutes_played
rating_decline_flag
next_sofascore_rating
```

Also drop the target you are not currently predicting. For example:

* If predicting `api_rating_decline_flag`, drop `next_api_rating`.
* If predicting `next_api_rating`, drop `api_rating_decline_flag`.

---

## Summary: Coverage & Training Guidance

| Group                     | Columns | Coverage            | Training Recommendation                                                          |
| ------------------------- | ------- | ------------------- | -------------------------------------------------------------------------------- |
| Match Identity            | 1–8     | 100%                | Use `date`, `competition`, `season`; drop IDs/high-cardinality labels as needed. |
| Player Identity           | 9–12    | 100%                | Encode `player_position`; be careful with `player_id = 0`.                       |
| API Player Stats          | 13–34   | 100%                | Strong base features. Treat `rating = 0` as missing.                             |
| Team Match Context        | 35–48   | mixed               | Use available team stats; impute missing competition-specific fields.            |
| Empty Opponent Stats      | 49–52   | 0%                  | Drop.                                                                            |
| SofaScore Source Features | 53–60   | sparse              | Keep only for secondary validation or PL-specific models.                        |
| FBRef Stats               | 61–72   | ~10%                | Optional richer subset or careful imputation.                                    |
| Squad Injury Context      | 73–78   | 66.28%              | Useful PL injury context; impute for non-PL rows.                                |
| Workload & Fatigue        | 79–85   | 90–100%             | Strong engineered predictors.                                                    |
| Lag Injury Features       | 86–96   | mixed               | Use count/binary columns; sparse text columns usually drop.                      |
| API Targets               | 98–100  | 67–100%             | Main modeling targets.                                                           |
| Legacy SofaScore Targets  | 101–102 | sparse / imbalanced | Secondary only.                                                                  |
| Next-Match Diagnostics    | 103–105 | 90–100%             | Diagnostics only; future information for pre-match modeling.                     |
